# DPM Engine · Processing

**Purpose.** Calculates the datapoints of one reference period for one or several frameworks and publishes the results for Regnology.

The run reads the parameterisation snapshot (`metadata_as_of`, resolved once at start-up), checks in `dp_lgl_reg_dpm_param_vld` that every requested framework has a `VALID` verdict issued after the latest change of its parameterisation, and then:

1. calculates the BASE variables from the Silver source tables, grouping the variables that share source table, period and grouping key so that each table is read once;
2. evaluates the formulas of the Calculated datapoints (DERIVED variables) natively in Spark;
3. builds the datapoints, consolidates them by view and entity, and publishes the Default values;
4. records the lineage of every variable and datapoint, the issues and the run status;
5. publishes the formatted values in `dp_lgl_reg_dpm_rpt_datapoint`, replacing in a single transaction the previous publication of the same period and frameworks.

Data issues (CALC domain) exclude only the affected values and the calculation continues; only technical or parameterisation failures set the run to `FAILED`. All result tables keep the full history by `exec_id`.

**Main parameters:** `reference_period` (YYYYMM or YYYYMMDD), `frameworks_csv`, `metadata_as_of` (`latest` or a timestamp), `run_id` (empty = generated), `run_mode` (`full` / `partial`), `req_id` (execution request that launches the run; empty otherwise).

**Prerequisites:** the engine tables exist (`DE_ENR_NB_DPM_Create_Tables`, run at deployment), and the parameterisation of every requested framework has been loaded and validated (`DE_ENR_NB_DPM_Metadata_Validation`): by the batch `DPM_Param_Load` after each DataLoader upload, or manually with `DE_ENR_NB_DPM_Metadata_Load`.

**Output:** the notebook ends with the run summary as its exit value (JSON), for the notebook or batch that launches it.

**Location:** workspace *DP FET Legal and Regulatory Reporting*, folder `01_Processes/DPM`, default Lakehouse `DP_FET_LH_LegalAndRegulatoryReporting`. All engine tables carry the prefix `dp_lgl_reg_dpm_`.


In [ ]:
%%configure -f
{
    "driverMemory": "7g",
    "driverCores": 2,
    "executorMemory": "7g",
    "executorCores": 2,
    "conf": {
        "spark.native.enabled": "true",
        "spark.dynamicAllocation.maxExecutors": "1"
    },
    "sessionTimeoutInSeconds": 7200
}

## 1. Parameters


In [ ]:
# Reference period of the run: YYYYMM (last day of the month) or YYYYMMDD.
reference_period = "20251231"
# Frameworks to calculate, separated by commas.
frameworks_csv = "FCP"

# Empty or "latest" = most recent parameterisation snapshot, resolved once at run start.
metadata_as_of = "latest"

run_id = ""  # Empty = generated as {FRAMEWORKS}_{YYYYMM}_{yyyyMMddTHHmmss} (UTC).
# "partial" only adds the suffix _P to the run identifier.
run_mode = "full"
# Execution request that launches the run (DE_ENR_NB_DPM_Request_Dispatcher). Empty when
# the run does not come from a request.
req_id = ""

# Maximum source variables per datapoint, and variables calculated per read of a table.
MAX_VARIABLES_PER_DATAPOINT = 70
max_variables_per_group = 250

# Engine tables.

datapoints_table = "dp_lgl_reg_dpm_param_datapoint"
filters_table = "dp_lgl_reg_dpm_param_fltr"
catalog_table = "dp_lgl_reg_dpm_param_ctlg"
issue_catalog_table = "dp_lgl_reg_dpm_param_iss_ctlg"
issue_log_table = "dp_lgl_reg_dpm_iss_log"
validation_table = "dp_lgl_reg_dpm_param_vld"

result_variable_table = "dp_lgl_reg_dpm_result_var_val"
result_datapoint_table = "dp_lgl_reg_dpm_result_datapoint_val"
lineage_variable_table = "dp_lgl_reg_dpm_lineage_var"
lineage_datapoint_table = "dp_lgl_reg_dpm_lineage_datapoint"
run_execution_table = "dp_lgl_reg_dpm_exec"
run_step_table = "dp_lgl_reg_dpm_exec_step"
failed_key_table = "dp_lgl_reg_dpm_failed_var_key"
# Default values published (append, history kept by exec_id).
datapoint_default_table = "dp_lgl_reg_dpm_datapoint_def"
# Publication table: only the latest run of each period and framework.
report_datapoint_table = "dp_lgl_reg_dpm_rpt_datapoint"
# Decimal separator of the published value. Final value pending confirmation with
# Regnology; the previous solution used the comma.
report_decimal_separator = ","

# Consolidation reference tables. dp_lgl_reg_dim_ent_hier is the Bank's entity
# hierarchy (replaces D_NIV_ENTIDADE); the other three replace FRP_ZAXIX_MAPA_COM_PAIS,
# FRP_TPDATAPOINT_MAPEAMENTO and FRP_TPMAPA_MAPEAMENTO of the previous solution. The two tables by template
# (ref_tmplt_ctry, ref_tmplt_csl_perim) are keyed by framework_cd + tmplt_cd, with the
# template written as in the Catalog tab (e.g. FINREP, 20.04).
entity_hierarchy_table = "dp_lgl_reg_dim_ent_hier"
ref_template_country_table = "dp_lgl_reg_dpm_ref_tmplt_ctry"
ref_source_datapoint_type_table = "dp_lgl_reg_dpm_ref_src_datapoint_typ"
ref_template_perimeter_table = "dp_lgl_reg_dpm_ref_tmplt_csl_perim"
# Schema/Lakehouse of the four reference tables. Empty = default Lakehouse of the notebook.
reference_tables_schema = ""

# Spark session settings.
aqe_enabled = "true"
shuffle_partitions = "8"


## Common definitions

Helpers shared with the other notebooks of the engine (`DE_ENR_NB_DPM_Common`).


In [ ]:
%run DE_ENR_NB_DPM_Common

## 2. Source / grain contract


In [ ]:
# Source table configuration, read from the SCD 2.2 table dp_lgl_reg_dpm_param_src_tbl
# (Sources tab of each parameterisation file). load_source_config() is called in the
# run cell, once the parameterisation snapshot has been resolved.
source_config_table = "dp_lgl_reg_dpm_param_src_tbl"


## 3. Common helpers, SCD resolution and expression compiler


In [ ]:
import ast
import hashlib
import json
import re
import traceback
import uuid
import calendar
from collections import Counter, defaultdict
from datetime import datetime, timezone
from functools import reduce
from typing import Any, Iterable

from notebookutils import mssparkutils
from pyspark import StorageLevel
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark.conf.set("spark.ms.autotune.enabled", "true")
spark.conf.set("spark.sql.session.timeZone", "UTC")
if str(aqe_enabled).strip():
    spark.conf.set("spark.sql.adaptive.enabled", str(aqe_enabled).strip().lower())
if str(shuffle_partitions).strip():
    spark.conf.set("spark.sql.shuffle.partitions", str(shuffle_partitions).strip())

class ClassifiedFailure(Exception):
    """Technical failure with its own code in the issue catalogue.

    The run cell raises `issue_code` with `params` instead of TECH.RUNTIME.UNCLASSIFIED.
    If `issue_code` is None, the issue has already been raised and the run only has to
    be set to FAILED.
    """

    def __init__(self, issue_code, message, params=None, **emit_fields):
        """Keeps the issue code, its parameters and the fields of the issue to record."""
        super().__init__(message)
        self.issue_code = issue_code
        self.params = dict(params or {})
        self.emit_fields = emit_fields


class MissingSourceColumn(Exception):
    """A metric column no longer exists in the source table."""

    def __init__(self, column):
        """Keeps the name of the missing column."""
        super().__init__(f"Column {column!r} not found in the source table")
        self.column = column


def stable_id(prefix, *parts):
    """Deterministic identifier: the prefix plus a hash of the given parts."""
    payload = "|".join(clean(part).upper() for part in parts)
    return f"{prefix}_{hashlib.sha256(payload.encode('utf-8')).hexdigest()[:24]}"


def reference_date(value):
    """ref_dt (TIMESTAMP) from the reference period: YYYYMMDD -> that day;
    YYYYMM -> last day of the month. Empty or invalid -> None."""
    digits = re.sub(r"\D", "", clean(value))
    try:
        if len(digits) == 8:
            return datetime(int(digits[:4]), int(digits[4:6]), int(digits[6:8]))
        if len(digits) == 6:
            year, month = int(digits[:4]), int(digits[4:6])
            return datetime(year, month, calendar.monthrange(year, month)[1])
    except ValueError:
        return None
    return None


def ref_dt_literal(value):
    """Literal ref_dt column used to write or filter the output tables. Built from text
    so that Spark interprets it in the session time zone (UTC), as the lineage query does."""
    moment = reference_date(value)
    if moment is None:
        return F.lit(None).cast("timestamp")
    return F.to_timestamp(F.lit(f"{moment:%Y-%m-%d %H:%M:%S}"))


# Strict date parsing: a value that does not match the pattern returns NULL.
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")


def build_filter_map(filter_rows):
    """Filter rows grouped by file, datapoint and variable."""
    result = defaultdict(list)
    for row in filter_rows:
        key = (
            clean(row.get("src_file_nm")),
            clean(row.get("datapoint_id")),
            clean(row.get("var_cd")).upper(),
        )
        result[key].append(row)
    return result


def compile_metric_expression(raw_expression, source_schema, default_aggregation):
    """Aggregation, value expression and columns of a metric of the parameterisation."""
    raw = clean(raw_expression)
    wrapper = re.fullmatch(
        r"\s*(SUM|COUNT|MIN|MAX|AVG|FIRST)\s*\((.*)\)\s*",
        raw,
        flags=re.I | re.S,
    )
    aggregation = default_aggregation
    body = raw
    if wrapper:
        aggregation = wrapper.group(1).upper()
        body = wrapper.group(2).strip()

    if aggregation == "COUNT" and body == "*":
        return {
            "aggregation": aggregation,
            "value_expression": f"CAST(1 AS {DECIMAL_SQL})",
            "metric_columns": (),
        }

    column_map = {name.upper(): name for name in source_schema}
    metric_columns = []

    def render_column(match):
        """Replaces a column of the metric with its numeric expression and records the column."""
        token = match.group(0)
        actual = column_map.get(token.upper())
        if actual is None:
            raise MissingSourceColumn(token)
        metric_columns.append(actual)
        # A NULL counts as 0 inside the metric (COALESCE per column), as in the previous
        # solution. Non-null values that cannot be converted are flagged as INVALID_VALUE.
        return (
            f"COALESCE(TRY_CAST({quote_identifier(actual)} AS {DECIMAL_SQL}), "
            f"CAST(0 AS {DECIMAL_SQL}))"
        )

    # Single-pass substitution: the generated text is not parsed again.
    compiled = re.sub(r"\b[A-Za-z_][A-Za-z0-9_]*\b", render_column, body)
    return {
        "aggregation": aggregation,
        "value_expression": compiled,
        "metric_columns": tuple(dict.fromkeys(metric_columns)),
    }


def normalize_having(raw):
    """HAVING condition written on the aggregated value column (val_amnt)."""
    value = clean(raw)
    if not value:
        return ""
    value = re.sub(r"SUM\s*\([^)]*\)", "val_amnt", value, flags=re.I)
    value = re.sub(r"\bMTDATAPOINT\b", "val_amnt", value, flags=re.I)
    # The former name of the value column is still accepted in HAVING conditions.
    value = re.sub(r"\bvalue_decimal\b", "val_amnt", value, flags=re.I)
    return value


def formula_variables(expression):
    """Variables (V1, V2...) used in a formula, in order of appearance."""
    return tuple(dict.fromkeys(v.upper() for v in LOCAL_VARIABLE_PATTERN.findall(clean(expression).upper())))


def arithmetic_sql(expression):
    """Render validated V1..Vn arithmetic to SQL plus a division-by-zero predicate."""
    tree = ast.parse(clean(expression).upper(), mode="eval")

    def render(node):
        """SQL text and division-by-zero predicate of one node of the formula."""
        if isinstance(node, ast.Expression):
            return render(node.body)
        if isinstance(node, ast.Name):
            return quote_identifier(node.id.upper()), "FALSE"
        if isinstance(node, ast.Constant):
            return str(node.value), "FALSE"
        if isinstance(node, ast.UnaryOp):
            value, div0 = render(node.operand)
            if isinstance(node.op, ast.USub):
                return f"(-({value}))", div0
            if isinstance(node.op, ast.UAdd):
                return f"(+({value}))", div0
        if isinstance(node, ast.BinOp):
            left, left_zero = render(node.left)
            right, right_zero = render(node.right)
            inherited = f"(({left_zero}) OR ({right_zero}))"
            if isinstance(node.op, ast.Add):
                return f"(({left}) + ({right}))", inherited
            if isinstance(node.op, ast.Sub):
                return f"(({left}) - ({right}))", inherited
            if isinstance(node.op, ast.Mult):
                return f"(({left}) * ({right}))", inherited
            if isinstance(node.op, ast.Div):
                zero = f"(({inherited}) OR (({right}) = CAST(0 AS {DECIMAL_SQL})))"
                value = (
                    f"(CASE WHEN ({right}) = CAST(0 AS {DECIMAL_SQL}) "
                    f"THEN NULL ELSE ({left}) / ({right}) END)"
                )
                return value, zero
        raise RuntimeError(f"Unsupported arithmetic node after validation: {ast.dump(node)}")

    return render(tree)


def key_json_expression(grain_columns):
    """Grouping key in JSON with the names of `grain_columns`. Null grain columns are
    omitted from the JSON."""
    return F.to_json(F.struct(*[
        F.col(grain_internal_column(key)).alias(key) for key, _ in grain_columns
    ]))


DEFAULT_VALUE_TYPE = "AMOUNT"


def normalize_value_type(raw):
    """Value type of a datapoint; an empty or unknown value becomes the default type."""
    text = re.sub(r"\s+", "", clean(raw).upper())
    return text if text in VALUE_TYPES else DEFAULT_VALUE_TYPE


def _decimal_separator(text_column):
    """Published text with the configured decimal separator."""
    separator = clean(report_decimal_separator) or "."
    if separator == ".":
        return text_column
    return F.regexp_replace(text_column, r"\.", separator)


def format_decimal_value(decimal_column, value_type_column):
    """Number -> published text according to the value type.

    Amount: 2 decimals, rounded. Percentage: 4 decimals, truncated. Integer: no
    decimals, rounded. Text: the number unformatted (validation only allows Text in
    Default datapoints). No thousands separator; report_decimal_separator as decimal
    separator.
    """
    value = decimal_column.cast(DECIMAL_SQL)
    scaled = value * F.lit(10000)
    truncated = F.when(value >= 0, F.floor(scaled)).otherwise(F.ceil(scaled))
    percentage = (truncated.cast(DECIMAL_SQL) / F.lit(10000)).cast("DECIMAL(38,4)")
    text = (
        F.when(value_type_column == "PERCENTAGE", percentage.cast("string"))
        .when(value_type_column == "INTEGER", F.round(value, 0).cast("DECIMAL(38,0)").cast("string"))
        .when(value_type_column == "TEXT", value.cast("string"))
        .otherwise(F.round(value, 2).cast("DECIMAL(38,2)").cast("string"))
    )
    return F.when(value.isNull(), F.lit(None).cast("string")).otherwise(_decimal_separator(text))


def format_default_value(raw_column, value_type_column):
    """A text Default is published as is; a numeric one is formatted with the same rules
    as a calculated value."""
    numeric = F.expr(f"TRY_CAST(TRIM(CAST({raw_column} AS STRING)) AS {DECIMAL_SQL})")
    return (
        F.when(
            (value_type_column == "TEXT") | numeric.isNull(),
            F.trim(F.col(raw_column).cast("string")),
        )
        .otherwise(format_decimal_value(numeric, value_type_column))
    )


def period_reference_date(physical_month):
    """Reference day of the period: YYYYMMDD -> that day; YYYYMM -> last day of the month."""
    digits = re.sub(r"\D", "", clean(physical_month))
    year, month = int(digits[:4]), int(digits[4:6])
    last_day = calendar.monthrange(year, month)[1]
    day = int(digits[6:8]) if len(digits) == 8 else last_day
    return datetime(year, month, day), last_day


def period_sql_predicate(period_column, data_type, physical_month, period_format=""):
    """Period filter of the lineage query, readable and equivalent to the engine's.

    With a declared format, a single condition is written in the format of the table
    (`periodo = '202512'`, `CAST(fecha AS DATE) = DATE'2025-12-31'`). The strict parsing of
    the engine (period_parsed_expr) guarantees that both select the same rows.
    """
    qcol = quote_identifier(period_column)
    reference, last_day = period_reference_date(physical_month)
    fmt = normalize_period_format(period_format)
    family = filter_type_family(data_type) if data_type is not None else "string"
    if fmt == PERIOD_FORMAT_DATE or (not fmt and family in {"date", "timestamp"}):
        return f"CAST({qcol} AS DATE) = DATE'{reference:%Y-%m-%d}'"
    if fmt:
        if not period_has_day(fmt) and reference.day != last_day:
            # Monthly table and a reference day that is not a month end: no rows are read.
            return f"FALSE /* {qcol} is monthly ({fmt}) and the reference day is not month end */"
        return f"TRIM(CAST({qcol} AS STRING)) = {sql_literal(format_period_value(reference, fmt))}"
    # No declared format (rows created before the Sources tab): the texts that
    # period_to_date_expr converts to that date.
    accepted = [f"{reference:%Y%m%d}"]
    if reference.day == last_day:
        accepted.append(f"{reference:%Y%m}")
    return (
        f"REGEXP_REPLACE(CAST({qcol} AS STRING), '[^0-9]', '') IN ("
        + ", ".join(sql_literal(value) for value in accepted) + ")"
    )


def build_source_sql(spec, period_column, source_schema):
    """Lineage query that reproduces the value of a BASE variable.

    If the aggregation is not FIRST, rows are grouped by the grain (Aggregated, or
    Calculated with an explicit aggregation); with FIRST the rows are returned ungrouped,
    as the engine reads them (one row per key; more raise CALC.CARDINALITY.MULTIPLE_ROWS).
    To see the detail of a grouped query: list the source columns and remove GROUP BY / HAVING.
    """
    actual = {name.upper(): name for name in source_schema}
    period_actual = actual.get(clean(period_column).upper(), clean(period_column))
    grain = []
    for key, physical in spec["grain_columns"]:
        name = actual.get(clean(physical).upper()) if clean(physical) else None
        if name:
            column = quote_identifier(name)
            grain.append(column if name == key else f"{column} AS {quote_identifier(key)}")
    grain_group = [item.split(" AS ")[0] for item in grain]

    aggregation = spec["agg_fcn_cd"]
    grouped = aggregation != "FIRST"
    if aggregation == "COUNT":
        value_sql = "COUNT(*)"
    elif grouped:
        value_sql = f"{aggregation}({spec['value_expression']})"
    else:
        value_sql = spec["value_expression"]

    lines = ["SELECT " + ",\n       ".join(grain + [f"{value_sql} AS val_amnt"])]
    lines.append(f"FROM {spec['phy_tbl_nm']}")
    lines.append(
        "WHERE " + period_sql_predicate(
            period_actual, source_schema.get(period_actual), spec["physical_month"],
            spec.get("per_fmt_cd", ""),
        )
    )
    if clean(spec["filter_expression"]) and clean(spec["filter_expression"]).upper() != "TRUE":
        lines.append(f"  AND ({spec['filter_expression']})")
    if grouped and grain_group:
        lines.append("GROUP BY " + ", ".join(grain_group))
    if grouped and clean(spec["having_ln"]):
        having = re.sub(r"\bval_amnt\b", value_sql, spec["having_ln"], flags=re.I)
        lines.append(f"HAVING {having}")
    return "\n".join(lines)


## 4. Issue handling and output tables


In [ ]:
from delta.tables import DeltaTable

def load_issue_catalog():
    """Active codes of the issue catalogue, by code."""
    frame = spark.table(issue_catalog_table)
    if "ctrl_state" in frame.columns:
        frame = frame.where(F.col("ctrl_state") == 1)
    if "row_present_flg" in frame.columns:
        frame = frame.where(F.col("row_present_flg") == 1)
    catalog = {}
    for row in frame.toLocalIterator():
        item = row.asDict(recursive=True)
        if clean(item.get("act_flg", "1")).upper() not in {"0", "FALSE", "F", "N", "NO", ""}:
            catalog[clean(item["iss_cd"])] = item
    if not catalog:
        raise RuntimeError("Issue catalog is empty.")
    return catalog


# Issue codes this notebook can raise. Checked at start-up: if any is missing from
# the issue catalogue, the run stops with a clear message.
REQUIRED_ISSUE_CODES = (
    "CALC.INPUT.NO_PERIOD_ROWS",
    "CALC.INPUT.PERIOD_UNPARSEABLE",
    "CALC.AGG.NO_CONTRIBUTING_ROWS",
    "CALC.INPUT.INVALID_VALUE",
    "CALC.INPUT.NO_VALID_VALUES",
    "CALC.INPUT.ALL_NULL_METRIC",
    "CALC.CARDINALITY.MULTIPLE_ROWS",
    "CALC.DERIVED.MISSING_DEPENDENCY",
    "CALC.DERIVED.DIVISION_BY_ZERO",
    "CALC.DERIVED.RESULT_INVALID",
    "CALC.DERIVED.ARITHMETIC_ERROR",
    "CALC.DEFAULT.NO_TARGET_KEYS",
    "META.SELECTION.EMPTY",
    "META.SELECTION.NOT_VALIDATED",
    "META.PERIOD.INVALID",
    "TECH.PARAM.INVALID",
    "TECH.SOURCE.READ_FAILED",
    "TECH.SOURCE.CONTRACT_CHANGED",
    "TECH.PERSISTENCE.FAILED",
    "TECH.RUNTIME.INVARIANT_FAILED",
    "TECH.RUNTIME.UNCLASSIFIED",
)


def check_issue_catalog(catalog):
    """Stops the run if an issue code this notebook can raise is missing from the catalogue."""
    missing = [code for code in REQUIRED_ISSUE_CODES if code not in catalog]
    if not missing:
        return
    detail = "missing issue codes: " + ", ".join(missing)
    if "TECH.ISSUE.CATALOG_INVALID" in catalog:
        raise ClassifiedFailure(
            "TECH.ISSUE.CATALOG_INVALID",
            f"Issue catalog is invalid: {detail}",
            params={"detail": detail},
        )
    raise RuntimeError(f"Issue catalog is invalid: {detail}")


def technical_detail(exc, limit=1000):
    """Error class and message, cut to `limit` characters."""
    return f"{type(exc).__name__}: {clean(str(exc))}"[:limit]


def reference_table_name(name):
    """Reference table with its explicit schema, if configured."""
    schema = clean(reference_tables_schema)
    return f"{schema}.{name}" if schema else name


def read_table_or_fail(table_name):
    """Read a source or reference table: an access failure is recorded as
    TECH.SOURCE.READ_FAILED instead of TECH.RUNTIME.UNCLASSIFIED."""
    try:
        return spark.table(table_name)
    except Exception as exc:
        raise ClassifiedFailure(
            "TECH.SOURCE.READ_FAILED",
            f"Unable to read source {table_name}: {technical_detail(exc)}",
            params={"src_tbl_id": table_name, "detail": technical_detail(exc)},
            source_table=table_name,
        ) from exc


def append_to_table(frame, table_name, artifact_type):
    """Append write: a write failure (permissions, concurrent writes...) is recorded as
    TECH.PERSISTENCE.FAILED."""
    try:
        frame.write.format("delta").mode("append").saveAsTable(table_name)
    except Exception as exc:
        raise ClassifiedFailure(
            "TECH.PERSISTENCE.FAILED",
            f"Unable to persist {artifact_type}: {technical_detail(exc)}",
            params={"artifact_type": artifact_type, "detail": technical_detail(exc)},
        ) from exc


def replace_in_table(frame, table_name, predicate, artifact_type):
    """Replace the rows matching `predicate` in a single commit (Delta replaceWhere).
    A concurrent write to the table is retried (write_with_retry); any other failure is
    recorded as TECH.PERSISTENCE.FAILED."""
    try:
        write_with_retry(
            lambda: (
                frame.write.format("delta")
                .mode("overwrite")
                .option("replaceWhere", predicate)
                .saveAsTable(table_name)
            )
        )
    except Exception as exc:
        raise ClassifiedFailure(
            "TECH.PERSISTENCE.FAILED",
            f"Unable to persist {artifact_type}: {technical_detail(exc)}",
            params={"artifact_type": artifact_type, "detail": technical_detail(exc)},
        ) from exc


class _SafeFormatDict(dict):
    """Prevents a missing template parameter from breaking the issue rendering."""

    def __missing__(self, key):
        """Leaves an unknown template parameter as {name}."""
        return "{" + key + "}"


class ProcessingIssues:
    def __init__(self, catalog, process_run_id, metadata_ts, ref_period):
        """Issue collector of one run."""
        self.catalog = catalog
        self.run_id = process_run_id
        self.metadata_ts = metadata_ts
        self.reference_period = ref_period
        self.rows = []

    def emit(
        self, issue_code, *, framework="", source_table="", datapoint_id="",
        variable_id="", group_id="", affected_count=0, params=None,
        raw_error_class="", raw_message="",
    ):
        """Adds one issue, with its message rendered from the catalogue template."""
        if issue_code not in self.catalog:
            raise KeyError(f"Issue code {issue_code!r} is not active in {issue_catalog_table}.")
        definition = self.catalog[issue_code]
        params = dict(params or {})
        rendered = clean(definition["msg_tmplt_ln"]).format_map(_SafeFormatDict(params))
        payload = "|".join([
            self.run_id, issue_code, framework, source_table, datapoint_id,
            variable_id, group_id, json.dumps(params, sort_keys=True, default=str),
        ])
        key = hashlib.sha256(payload.encode("utf-8")).hexdigest()
        self.rows.append({
            "iss_id": key, "exec_id": self.run_id, "vld_id": "", "phs_cd": "PROCESSING",
            "param_snpsh_dt": self.metadata_ts,
            "ref_dt": None,  # filled in persist() with ref_dt_literal()
            "iss_cd": issue_code, "sev_cd": clean(definition["sev_cd"]).upper(),
            "framework_cd": framework, "src_tbl_id": source_table,
            "datapoint_id": datapoint_id, "var_id": variable_id,
            "grp_id": group_id, "afct_nr": int(affected_count or 0),
            "param_json_ln": json.dumps(params, sort_keys=True, separators=(",", ":"), default=str),
            "msg_ln": rendered,
            "raw_err_cla_nm": raw_error_class,
            "raw_msg_ln": clean(raw_message)[:4000],
            "crtn_dt": utc_now(),
        })

    def persist(self):
        """Writes the collected issues in the issue log."""
        if self.rows:
            schema = spark.table(issue_log_table).schema
            values = [
                tuple(row.get(field.name) for field in schema.fields)
                for row in self.rows
            ]
            (
                spark.createDataFrame(values, schema)
                .withColumn("ref_dt", ref_dt_literal(self.reference_period))
                .write.format("delta").mode("append").saveAsTable(issue_log_table)
            )


def append_step(run_id_value, step_name, status, row_count=0, group_id="", message="", started=None):
    """Records one step of the run in the step table."""
    row = [{
        "exec_id": run_id_value, "step_nm": step_name, "grp_id": group_id,
        "step_stts": status, "row_nr": int(row_count or 0), "msg_ln": clean(message),
        "start_dt": started or utc_now(), "end_dt": utc_now(),
    }]
    spark.createDataFrame(row).write.format("delta").mode("append").saveAsTable(run_step_table)


def frame_from_table_schema(table_name, row):
    """One-row DataFrame with the schema of the given table."""
    schema = spark.table(table_name).schema
    values = [tuple(row.get(field.name) for field in schema.fields)]
    return spark.createDataFrame(values, schema)


def build_run_id(requested, reference_period_raw, mode, started):
    """Readable run identifier, timestamped in UTC:
    {FRAMEWORKS}_{YYYYMM}_{yyyyMMddTHHmmss}[_P]. If it already exists, -2, -3... is added."""
    label = "+".join(requested) if requested else "ALL"
    period = re.sub(r"\D", "", clean(reference_period_raw))[:6] or "NOPERIOD"
    base = f"{label}_{period}_{started:%Y%m%dT%H%M%S}"
    if clean(mode).lower() == "partial":
        base += "_P"
    existing = {
        row["exec_id"]
        for row in spark.table(run_execution_table)
        .where(F.col("exec_id").startswith(base))
        .select("exec_id")
        .toLocalIterator()
    }
    candidate, suffix = base, 2
    while candidate in existing:
        candidate = f"{base}-{suffix}"
        suffix += 1
    return candidate


def upsert_run_execution(row):
    # ref_dt arrives as the reference period (text) and is converted in Spark.
    """Run row inserted, or updated if it exists, in the run table. A concurrent write to
    the table is retried (write_with_retry)."""
    reference = row.get("ref_dt")
    source = frame_from_table_schema(run_execution_table, {**row, "ref_dt": None}).withColumn(
        "ref_dt", ref_dt_literal(reference)
    )
    write_with_retry(
        lambda: (
            DeltaTable.forName(spark, run_execution_table)
            .alias("target")
            .merge(source.alias("source"), "target.exec_id = source.exec_id")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )
    )


## 5. Build validated execution plan


In [ ]:
def select_metadata_snapshot(metadata_ts, requested_frameworks):
    """Datapoints, filters and catalogue of the snapshot, limited to the requested frameworks."""
    datapoints = scd_snapshot(datapoints_table, metadata_ts)
    filters = scd_snapshot(filters_table, metadata_ts)
    catalog = scd_snapshot(catalog_table, metadata_ts)

    if requested_frameworks:
        datapoints = datapoints.where(F.upper("framework_cd").isin(list(requested_frameworks)))
        files = [row["src_file_nm"] for row in datapoints.select("src_file_nm").distinct().toLocalIterator()]
        filters = filters.where(F.col("src_file_nm").isin(files))
        catalog = catalog.where(F.col("src_file_nm").isin(files))
    return datapoints, filters, catalog


def parameter_changes(frame, metadata_ts):
    """Version starts and ends of the given parameterisation rows, up to the snapshot."""
    return (
        frame.select(F.explode(F.array("ref_start_dt", "ref_end_dt")).alias("chg_dt"))
        .where(F.col("chg_dt") <= F.lit(metadata_ts))
    )


def framework_last_change(framework, metadata_ts):
    """Latest change of the parameterisation of a framework, up to the snapshot.

    It is the latest version start or end of its rows in the four parameterisation tables,
    so new, modified and removed rows all count. Filters, Catalog and Sources belong to a
    framework through their file; the Sources rows without a file (configuration that
    existed before the Sources tab) count for every framework. None if it has no rows.
    """
    datapoints = spark.table(datapoints_table).where(F.upper("framework_cd") == framework)
    files = [
        row["src_file_nm"]
        for row in datapoints.select("src_file_nm").distinct().toLocalIterator()
    ]
    sources = spark.table(source_config_table).where(
        F.coalesce(F.col("src_file_nm"), F.lit("")).isin(files + [""])
    )
    changes = [
        parameter_changes(frame, metadata_ts)
        for frame in (
            datapoints,
            spark.table(filters_table).where(F.col("src_file_nm").isin(files)),
            spark.table(catalog_table).where(F.col("src_file_nm").isin(files)),
            sources,
        )
    ]
    return (
        reduce(lambda left, right: left.unionByName(right), changes)
        .agg(F.max("chg_dt"))
        .first()[0]
    )


def frameworks_without_verdict(frameworks, metadata_ts):
    """Frameworks that cannot be calculated with the snapshot.

    A framework can be calculated when it has a VALID verdict issued for a snapshot that is
    not earlier than the latest change of its parameterisation and not later than the
    snapshot of the run. A load that only changes other frameworks creates a new snapshot
    but does not invalidate its verdict.
    """
    verdicts = (
        spark.table(validation_table)
        .where(F.col("vld_stts") == "VALID")
        .where(F.col("param_snpsh_dt") <= F.lit(metadata_ts))
    )
    missing = []
    for framework in frameworks:
        changed = framework_last_change(framework, metadata_ts)
        validated = changed is not None and (
            verdicts.where(F.col("framework_cd") == framework)
            .where(F.col("param_snpsh_dt") >= F.lit(changed))
            .first()
            is not None
        )
        if not validated:
            missing.append(framework)
    return missing


def require_validated_snapshot(frameworks, metadata_ts, issues):
    """Stops the run if a framework has no usable VALID verdict, with one
    META.SELECTION.NOT_VALIDATED issue per framework."""
    not_validated = frameworks_without_verdict(frameworks, metadata_ts)
    for framework in not_validated:
        issues.emit(
            "META.SELECTION.NOT_VALIDATED",
            framework=framework,
            params={"param_snpsh_dt": str(metadata_ts), "framework_cd": framework},
        )
    if not_validated:
        raise ClassifiedFailure(
            None,
            f"Metadata snapshot {metadata_ts} is not VALID for framework(s): "
            f"{', '.join(not_validated)}.",
        )


def compile_plan(datapoints, filters, catalog, reference_period_value):
    """Execution plan: BASE and DERIVED variables, mappings, defaults, value types and lineage items."""
    dp_rows = frame_records(datapoints)
    filter_rows = frame_records(filters)
    catalog_rows = frame_records(catalog)

    # Schema per physical table, read once per run.
    schema_cache = {}

    def source_schema_of(physical_table):
        """Schema of a physical table, read once per run."""
        if physical_table not in schema_cache:
            schema_cache[physical_table] = {
                field.name: field.dataType
                for field in read_table_or_fail(physical_table).schema.fields
            }
        return schema_cache[physical_table]

    filter_map = build_filter_map(filter_rows)
    catalog_map = {
        (clean(row.get("src_file_nm")), clean(row.get("datapoint_id"))): row
        for row in catalog_rows
    }

    grouped = defaultdict(list)
    for row in dp_rows:
        grouped[
            (
                clean(row.get("src_file_nm")),
                clean(row.get("framework_cd")).upper(),
                clean(row.get("datapoint_id")),
            )
        ].append(row)

    base_specs = []
    derived_specs = []
    mapping_rows = []
    # Default datapoints (literal value, no table is read).
    default_items = []
    # Value type of each selected datapoint (publication format).
    value_type_rows = []
    # Output of each datapoint (type, variables, periods and grains), used to expand
    # the Rollups at the end, once all their components have been compiled.
    datapoint_outputs = {}
    rollup_items = []
    # Data for the datapoint lineage (formula, variables with their sign and query).
    declared_types = {}
    formula_info = {}
    rollup_expressions = {}
    var_info = {}

    for (source_file, framework, datapoint_id), rows in sorted(grouped.items()):
        row_type = clean(rows[0].get("datapoint_cl_typ")).upper()
        catalog_entry = catalog_map.get((source_file, datapoint_id), {})
        declared_types[(framework, datapoint_id)] = row_type
        value_type_rows.append({
            "framework_cd": framework,
            "datapoint_id": datapoint_id,
            "val_typ": normalize_value_type(catalog_entry.get("val_typ")),
        })
        if row_type == "DEFAULT":
            # Only the value of Metric / Formula is used. Validation guarantees that there is
            # no Source Table, Variable, Having, Period or Filters.
            default_items.append({
                "framework_cd": framework,
                "datapoint_id": datapoint_id,
                "tmplt_cd": clean(catalog_entry.get("tmplt_cd")),
                "row_cd": clean(catalog_entry.get("row_cd")),
                "column_cd": clean(catalog_entry.get("column_cd")),
                "def_val_ln": clean(rows[0].get("metric_fmla_ln")),
            })
            continue
        if row_type == "ROLLUP":
            rollup_items.append((source_file, framework, datapoint_id, rows[0]))
            rollup_expressions[(framework, datapoint_id)] = clean(rows[0].get("metric_fmla_ln"))
            continue
        source_rows = [row for row in rows if clean(row.get("src_tbl_id"))]
        formula_rows = [row for row in rows if not clean(row.get("src_tbl_id"))]
        catalog_row = catalog_map[(source_file, datapoint_id)]

        local_to_internal = {}
        current_base_specs = []
        for index, row in enumerate(source_rows, 1):
            original_variable = clean(row.get("var_cd")).upper()
            local_variable = (
                original_variable
                if original_variable
                else ("V1" if len(source_rows) == 1 else "")
            )
            logical_source = clean(row.get("src_tbl_id")).upper()
            # Each parameterisation file declares its source tables in the Sources tab.
            config = source_config_for(source_file, logical_source)
            if config is None:
                raise ClassifiedFailure(
                    "TECH.SOURCE.CONTRACT_CHANGED",
                    f"Source {logical_source} is not configured in the Sources sheet of {source_file}.",
                    params={
                        "src_tbl_id": logical_source,
                        "detail": f"not configured in the Sources sheet of {source_file}",
                    },
                    source_table=logical_source, framework=framework, datapoint_id=datapoint_id,
                )
            physical_table = clean(config["phy_tbl_nm"])
            source_schema = source_schema_of(physical_table)
            try:
                metric = compile_metric_expression(
                    row.get("metric_fmla_ln"),
                    source_schema,
                    "SUM" if row_type == "AGGREGATED" else "FIRST",
                )
            except MissingSourceColumn as exc:
                detail = f"metric column {exc.column} of {datapoint_id}/{local_variable} no longer exists"
                raise ClassifiedFailure(
                    "TECH.SOURCE.CONTRACT_CHANGED",
                    f"Source {physical_table} no longer satisfies the validated contract: {detail}",
                    params={"src_tbl_id": physical_table, "detail": detail},
                    source_table=physical_table, framework=framework, datapoint_id=datapoint_id,
                ) from exc
            row_filters = filter_map.get(
                (source_file, datapoint_id, original_variable),
                [],
            )
            schema_upper = {name.upper() for name in source_schema}
            for item in row_filters:
                filter_column = clean(item.get("src_column_nm"))
                if filter_column.upper() not in schema_upper:
                    detail = f"filter column {filter_column} of {datapoint_id}/{local_variable} no longer exists"
                    raise ClassifiedFailure(
                        "TECH.SOURCE.CONTRACT_CHANGED",
                        f"Source {physical_table} no longer satisfies the validated contract: {detail}",
                        params={"src_tbl_id": physical_table, "detail": detail},
                        source_table=physical_table, framework=framework, datapoint_id=datapoint_id,
                    )
            filter_sql = " AND ".join(
                f"({build_filter_sql(item, source_schema)})"
                for item in row_filters
            ) or "TRUE"
            try:
                period = resolve_period(reference_period_value, row.get("per_expr_cd"))
            except UnsupportedPeriod as exc:
                object_ref = f"{datapoint_id}/{local_variable or original_variable}"
                raise ClassifiedFailure(
                    "META.PERIOD.INVALID",
                    str(exc),
                    params={"period_value": clean(row.get("per_expr_cd")), "object_ref": object_ref},
                    framework=framework, datapoint_id=datapoint_id,
                ) from exc
            variable_id = stable_id(
                "VAR", source_file, framework, datapoint_id, local_variable
            )
            spec = {
                "metadata_source_file": source_file,
                "framework_cd": framework,
                "datapoint_id": datapoint_id,
                "var_cd": local_variable,
                "var_id": variable_id,
                "var_cl_typ": "BASE",
                "src_tbl_id": logical_source,
                "phy_tbl_nm": physical_table,
                "grain_id": clean(config["grain_id"]),
                "group_columns": tuple(config["group_columns"]),
                "grain_columns": tuple(config["grain_columns"]),
                # Period column and format of the source table.
                "per_column_nm": clean(config["per_column_nm"]),
                "per_fmt_cd": config["per_fmt_cd"],
                "per_cd": period["per_cd"],
                "per_frq_cd": period["per_frq_cd"],
                "physical_month": period["physical_month"],
                "agg_fcn_cd": metric["aggregation"],
                "value_expression": metric["value_expression"],
                "metric_columns": metric["metric_columns"],
                "metric_fmla_ln": clean(row.get("metric_fmla_ln")),
                "filter_expression": filter_sql,
                "having_ln": normalize_having(row.get("having_ln")),
                "fmla_ln": "",
                "dependency_json_ln": "[]",
            }
            # Complete SELECT for the lineage (filter_expression remains the WHERE clause used
            # by the engine).
            spec["source_sql"] = build_source_sql(spec, config["per_column_nm"], source_schema)
            base_specs.append(spec)
            current_base_specs.append(spec)
            local_to_internal[local_variable] = variable_id
            var_info[variable_id] = {
                "var_cd": local_variable, "datapoint_id": datapoint_id,
                "src_tbl_id": logical_source,
                "per_cd": spec["per_cd"], "per_frq_cd": spec["per_frq_cd"],
            }

        if row_type == "AGGREGATED":
            for spec in current_base_specs:
                mapping_rows.append({
                    "metadata_source_file": source_file,
                    "framework_cd": framework,
                    "datapoint_id": datapoint_id,
                    "var_id": spec["var_id"],
                    "metadata_type": row_type,
                    "src_tbl_id": spec["src_tbl_id"],
                    "tmplt_cd": clean(catalog_row.get("tmplt_cd")),
                    "row_cd": clean(catalog_row.get("row_cd")),
                    "column_cd": clean(catalog_row.get("column_cd")),
                    "sign": 1,
                })
            datapoint_outputs[(source_file, datapoint_id)] = {
                "datapoint_cl_typ": row_type,
                "variables": [(spec["var_id"], spec["src_tbl_id"]) for spec in current_base_specs],
                "periods": {spec["per_cd"] for spec in current_base_specs},
                "grains": {spec["grain_id"] for spec in current_base_specs},
            }

        elif formula_rows:
            formula = clean(formula_rows[0].get("metric_fmla_ln")).upper()
            derived_id = stable_id("CALC", source_file, framework, datapoint_id)
            first = current_base_specs[0]
            derived_specs.append({
                "metadata_source_file": source_file,
                "framework_cd": framework,
                "datapoint_id": datapoint_id,
                "var_id": derived_id,
                "var_cd": "RESULT",
                "var_cl_typ": "DERIVED",
                "grain_id": first["grain_id"],
                "per_cd": first["per_cd"],
                "per_frq_cd": first["per_frq_cd"],
                "fmla_ln": formula,
                "local_to_internal": dict(local_to_internal),
                "required_locals": formula_variables(formula),
                "dependency_json_ln": json.dumps(local_to_internal, sort_keys=True),
            })
            formula_info[(framework, datapoint_id)] = {
                "fmla_ln": formula,
                "local_to_internal": {
                    local: internal for local, internal in local_to_internal.items()
                    if local in formula_variables(formula)
                },
            }
            var_info[derived_id] = {
                "var_cd": "RESULT", "datapoint_id": datapoint_id,
                "per_cd": first["per_cd"], "per_frq_cd": first["per_frq_cd"],
            }
            mapping_rows.append({
                "metadata_source_file": source_file,
                "framework_cd": framework,
                "datapoint_id": datapoint_id,
                "var_id": derived_id,
                "metadata_type": row_type,
                "src_tbl_id": "",
                "tmplt_cd": clean(catalog_row.get("tmplt_cd")),
                "row_cd": clean(catalog_row.get("row_cd")),
                "column_cd": clean(catalog_row.get("column_cd")),
                "sign": 1,
            })
            datapoint_outputs[(source_file, datapoint_id)] = {
                "datapoint_cl_typ": row_type,
                "variables": [(derived_id, "")],
                "periods": {first["per_cd"]},
                "grains": {first["grain_id"]},
            }
        else:
            # Single-source Calculated maps its BASE variable directly.
            spec = current_base_specs[0]
            mapping_rows.append({
                "metadata_source_file": source_file,
                "framework_cd": framework,
                "datapoint_id": datapoint_id,
                "var_id": spec["var_id"],
                "metadata_type": row_type,
                "src_tbl_id": spec["src_tbl_id"],
                "tmplt_cd": clean(catalog_row.get("tmplt_cd")),
                "row_cd": clean(catalog_row.get("row_cd")),
                "column_cd": clean(catalog_row.get("column_cd")),
                "sign": 1,
            })
            datapoint_outputs[(source_file, datapoint_id)] = {
                "datapoint_cl_typ": row_type,
                "variables": [(spec["var_id"], spec["src_tbl_id"])],
                "periods": {spec["per_cd"]},
                "grains": {spec["grain_id"]},
            }

    # Rollup expansion: the total receives the variables of its components with their
    # sign and inherits their type (Aggregated -> consolidated by views; Calculated ->
    # published by key). Validation rejects invalid cases beforehand; this is a safety net.
    for source_file, framework, datapoint_id, row in rollup_items:
        def rollup_error(detail):
            """Invariant failure of the current Rollup, with the given detail."""
            return ClassifiedFailure(
                "TECH.RUNTIME.INVARIANT_FAILED",
                f"Rollup {datapoint_id}: {detail}",
                params={"stage": "COMPILE_PLAN", "detail": f"Rollup {datapoint_id}: {detail}"},
                framework=framework, datapoint_id=datapoint_id,
            )

        catalog_row = catalog_map.get((source_file, datapoint_id))
        if catalog_row is None:
            raise rollup_error("datapoint not found in the catalog")
        try:
            terms = parse_rollup_expression(row.get("metric_fmla_ln"))
        except ValueError as exc:
            raise rollup_error(str(exc)) from exc
        children = []
        for child_id, sign in terms:
            child = datapoint_outputs.get((source_file, child_id))
            if child is None:
                raise rollup_error(f"child datapoint {child_id} not found or not calculable")
            children.append((child_id, sign, child))
        types = {child["datapoint_cl_typ"] for _, _, child in children}
        if len(types) != 1:
            raise rollup_error("mixes AGGREGATED and CALCULATED children")
        rollup_type = next(iter(types))
        periods = set().union(*(child["periods"] for _, _, child in children))
        if len(periods) != 1:
            raise rollup_error(f"children with different periods {sorted(periods)}")
        if rollup_type != "AGGREGATED":
            grains = set().union(*(child["grains"] for _, _, child in children))
            if len(grains) != 1:
                raise rollup_error(f"CALCULATED children with different grains {sorted(grains)}")
        for child_id, sign, child in children:
            for variable_id, child_source_table in child["variables"]:
                mapping_rows.append({
                    "metadata_source_file": source_file,
                    "framework_cd": framework,
                    "datapoint_id": datapoint_id,
                    "var_id": variable_id,
                    "metadata_type": rollup_type,
                    "src_tbl_id": child_source_table,
                    "tmplt_cd": clean(catalog_row.get("tmplt_cd")),
                    "row_cd": clean(catalog_row.get("row_cd")),
                    "column_cd": clean(catalog_row.get("column_cd")),
                    "sign": sign,
                })

    datapoint_lineage_items = build_datapoint_lineage_items(
        mapping_rows, default_items, declared_types, formula_info,
        rollup_expressions, var_info,
    )

    return {
        "base_specs": base_specs,
        "derived_specs": derived_specs,
        "mapping_rows": mapping_rows,
        "default_items": default_items,
        "value_type_rows": value_type_rows,
        "datapoint_lineage_items": datapoint_lineage_items,
    }


def build_datapoint_lineage_items(
    mapping_rows, default_items, declared_types, formula_info, rollup_expressions, var_info
):
    """One entry per datapoint with what the datapoint lineage needs.

    - fmla_ln: how it is calculated (V1, V1 + V2, the formula of a Calculated datapoint,
      the expression of a Rollup or the value of a Default).
    - variables: variables used, with their sign (+1 / -1 in Rollups).
    - csl_flg: 1 if consolidated by views (Aggregated and Rollup of Aggregated).
    """
    by_datapoint = defaultdict(list)
    for row in mapping_rows:
        by_datapoint[(row["framework_cd"], row["datapoint_id"])].append(row)

    items = []
    for (framework, datapoint_id), rows in sorted(by_datapoint.items()):
        first = rows[0]
        declared = declared_types.get((framework, datapoint_id), first["metadata_type"])
        formula_entry = formula_info.get((framework, datapoint_id))
        if formula_entry and declared != "ROLLUP":
            # Calculated with a formula: the variables of the formula, not the RESULT variable.
            variables = [
                {
                    "datapoint_id": datapoint_id, "var_id": internal, "var_cd": local,
                    "sign": 1, "src_tbl_id": var_info.get(internal, {}).get("src_tbl_id", ""),
                }
                for local, internal in sorted(formula_entry["local_to_internal"].items())
            ]
        else:
            variables = []
            for row in rows:
                info = var_info.get(row["var_id"], {})
                variables.append({
                    "datapoint_id": info.get("datapoint_id", datapoint_id),
                    "var_id": row["var_id"],
                    "var_cd": info.get("var_cd", ""),
                    "sign": int(row["sign"]),
                    "src_tbl_id": row["src_tbl_id"],
                })
        first_info = var_info.get(first["var_id"], {})
        if declared == "ROLLUP":
            formula = rollup_expressions.get((framework, datapoint_id), "")
        elif formula_entry:
            formula = formula_entry["fmla_ln"]
        else:
            formula = " + ".join(item["var_cd"] for item in variables)
        items.append({
            "framework_cd": framework,
            "datapoint_id": datapoint_id,
            "tmplt_cd": first["tmplt_cd"],
            "row_cd": first["row_cd"],
            "column_cd": first["column_cd"],
            "datapoint_cl_typ": declared,
            "fmla_ln": formula,
            "variables": variables,
            "formula_vars": (formula_entry or {}).get("local_to_internal") if declared != "ROLLUP" else None,
            "csl_flg": 1 if first["metadata_type"] == "AGGREGATED" else 0,
            "per_cd": first_info.get("per_cd", ""),
            "per_frq_cd": first_info.get("per_frq_cd", ""),
        })
    for item in default_items:
        items.append({
            "framework_cd": item["framework_cd"],
            "datapoint_id": item["datapoint_id"],
            "tmplt_cd": item["tmplt_cd"],
            "row_cd": item["row_cd"],
            "column_cd": item["column_cd"],
            "datapoint_cl_typ": "DEFAULT",
            "fmla_ln": item["def_val_ln"],
            "variables": [],
            "formula_vars": None,
            "csl_flg": 0,
            "per_cd": "",
            "per_frq_cd": "",
        })
    return items


def execution_groups(base_specs):
    """BASE variables grouped by source table, period, grain and period column, in bounded blocks."""
    grouped = defaultdict(list)
    for spec in base_specs:
        key = (
            spec["phy_tbl_nm"],
            spec["physical_month"],
            spec["grain_id"],
            spec["grain_columns"],
            # Two files may read the same table with a different period column or format;
            # they do not share the read.
            spec["per_column_nm"],
            spec["per_fmt_cd"],
        )
        grouped[key].append(spec)

    groups = []
    for key, specs in sorted(grouped.items(), key=lambda item: str(item[0])):
        for block_index, start in enumerate(
            range(0, len(specs), int(max_variables_per_group)),
            1,
        ):
            groups.append({
                "key": key,
                # Same identifier as before the period column and format were added.
                "grp_id": stable_id("GROUP", *key[:4], block_index),
                "phy_tbl_nm": key[0],
                "physical_month": key[1],
                "grain_id": key[2],
                "grain_columns": key[3],
                "group_columns": tuple(name for name, _ in key[3]),
                "per_column_nm": key[4],
                "per_fmt_cd": key[5],
                "specs": specs[start:start + int(max_variables_per_group)],
            })
    return groups


## 6. BASE variables


In [ ]:
# Rows whose period cannot be read with the declared format. Counted once per table,
# column and format in each run (across all periods: an unreadable row could belong
# to the period being calculated).
UNPARSEABLE_EXAMPLES = 3


def report_unparseable_periods(source, table_name, period_column, period_format, issues, checked):
    """Raises CALC.INPUT.PERIOD_UNPARSEABLE once per table, column and format, if rows cannot be read."""
    check_key = (table_name, period_column.upper(), period_format)
    if check_key in checked:
        return
    checked.add(check_key)
    if not period_format or period_format == PERIOD_FORMAT_DATE:
        return
    bad = source.where(period_unparseable_expr(period_column, period_format))
    count = bad.count()
    if not count:
        return
    examples = [
        row["v"] for row in
        bad.select(period_text_expr(period_column).alias("v")).distinct()
        .orderBy("v").take(UNPARSEABLE_EXAMPLES)
    ]
    issues.emit(
        "CALC.INPUT.PERIOD_UNPARSEABLE",
        source_table=table_name,
        affected_count=count,
        params={
            "src_tbl_id": table_name,
            "src_column_nm": period_column,
            "per_fmt_cd": period_format,
            "afct_nr": count,
            "examples": ", ".join(repr(value) for value in examples),
        },
    )


def period_source_frame(table_name, physical_month, period_column_cfg, period_format,
                        cache, issues, unparseable_checked):
    """Rows of the source table for the period. The period column and format come from
    the group (Sources tab of the datapoint's file)."""
    key = (table_name, physical_month, clean(period_column_cfg).upper(), period_format)
    if key in cache:
        return cache[key]

    source = read_table_or_fail(table_name)
    schema_map = {name.upper(): name for name in source.columns}
    configured_period = clean(period_column_cfg)
    if configured_period.upper() not in schema_map:
        # The validated period column no longer exists in the source table.
        detail = f"period column {configured_period} no longer exists"
        raise ClassifiedFailure(
            "TECH.SOURCE.CONTRACT_CHANGED",
            f"Source {table_name} no longer satisfies the validated contract: {detail}",
            params={"src_tbl_id": table_name, "detail": detail},
            source_table=table_name,
        )
    period_column = schema_map[configured_period.upper()]

    source_period = period_to_date_expr(
        period_column,
        source.schema[period_column].dataType,
        period_format,
    )
    reference_period_date = period_literal_to_date(physical_month)

    try:
        report_unparseable_periods(
            source, table_name, period_column, period_format, issues, unparseable_checked
        )
    except ClassifiedFailure:
        raise
    except Exception as exc:
        raise ClassifiedFailure(
            "TECH.SOURCE.READ_FAILED",
            f"Unable to read source {table_name}: {technical_detail(exc)}",
            params={"src_tbl_id": table_name, "detail": technical_detail(exc)},
            source_table=table_name,
        ) from exc

    frame = (
        source.where(source_period == reference_period_date)
        .persist(StorageLevel.MEMORY_AND_DISK)
    )
    # Force materialization inside the scope that owns this cached source.
    try:
        frame.count()
    except Exception as exc:
        frame.unpersist()
        raise ClassifiedFailure(
            "TECH.SOURCE.READ_FAILED",
            f"Unable to read source {table_name}: {technical_detail(exc)}",
            params={"src_tbl_id": table_name, "detail": technical_detail(exc)},
            source_table=table_name,
        ) from exc
    cache[key] = frame
    return frame


def invalid_numeric_predicate(spec):
    """Condition true for rows with a non-null metric value that is not numeric."""
    predicates = [
        F.col(column).isNotNull()
        & F.expr(
            f"TRY_CAST({quote_identifier(column)} AS {DECIMAL_SQL})"
        ).isNull()
        for column in spec["metric_columns"]
    ]
    if not predicates:
        return F.lit(False)
    return reduce(lambda left, right: left | right, predicates)


def build_base_group_frame(source, group):
    """Aggregates of all the variables of a group in one read of the source: one row per variable and key."""
    aggregate_expressions = []
    aliases = []

    for index, spec in enumerate(group["specs"]):
        prefix = f"m{index:04d}"
        aliases.append((prefix, spec))

        condition = F.expr(spec["filter_expression"])
        metric = F.expr(spec["value_expression"])
        invalid = invalid_numeric_predicate(spec)
        valid = condition & (~invalid) & metric.isNotNull()

        matched_count = F.sum(
            F.when(condition, F.lit(1)).otherwise(F.lit(0))
        ).cast("long")
        # Rows matching the filter with ALL metric columns NULL. If all the rows of a key are
        # like this, 0 is published and a warning is raised.
        if spec["metric_columns"]:
            all_null = reduce(
                lambda left, right: left & right,
                [F.col(column).isNull() for column in spec["metric_columns"]],
            )
            all_null_count = F.sum(
                F.when(condition & all_null, F.lit(1)).otherwise(F.lit(0))
            ).cast("long")
        else:
            all_null_count = F.lit(0).cast("long")
        invalid_count = F.sum(
            F.when(condition & invalid, F.lit(1)).otherwise(F.lit(0))
        ).cast("long")

        if spec["agg_fcn_cd"] == "COUNT":
            valid_count = matched_count
            value = matched_count.cast(DECIMAL_SQL)
        else:
            valid_count = F.sum(
                F.when(valid, F.lit(1)).otherwise(F.lit(0))
            ).cast("long")

            candidate = F.when(valid, metric)
            aggregation = spec["agg_fcn_cd"]
            if aggregation == "SUM":
                value = F.sum(candidate).cast(DECIMAL_SQL)
            elif aggregation == "MIN":
                value = F.min(candidate).cast(DECIMAL_SQL)
            elif aggregation == "MAX":
                value = F.max(candidate).cast(DECIMAL_SQL)
            elif aggregation == "AVG":
                value = F.avg(candidate).cast(DECIMAL_SQL)
            elif aggregation == "FIRST":
                value = F.first(candidate, ignorenulls=True).cast(DECIMAL_SQL)
            else:
                raise RuntimeError(
                    f"Unsupported aggregation after validation: {aggregation}"
                )

        aggregate_expressions.extend([
            value.alias(prefix + "_value"),
            matched_count.alias(prefix + "_matched"),
            valid_count.alias(prefix + "_valid"),
            invalid_count.alias(prefix + "_invalid"),
            all_null_count.alias(prefix + "_allnull"),
        ])

    grain_internal = [grain_internal_column(key) for key, _ in group["grain_columns"]]
    aggregated = source.groupBy(*grain_internal).agg(*aggregate_expressions)

    structs = []
    for prefix, spec in aliases:
        structs.append(
            F.struct(
                F.lit(spec["framework_cd"]).alias("framework_cd"),
                F.lit(spec["datapoint_id"]).alias("datapoint_id"),
                F.lit(spec["var_id"]).alias("var_id"),
                F.lit(spec["var_cd"]).alias("var_cd"),
                F.lit(spec["src_tbl_id"]).alias("src_tbl_id"),
                F.lit(spec["grain_id"]).alias("grain_id"),
                F.lit(spec["per_cd"]).alias("per_cd"),
                F.lit(spec["per_frq_cd"]).alias("per_frq_cd"),
                F.lit(spec["agg_fcn_cd"]).alias("agg_fcn_cd"),
                F.lit(spec["metric_fmla_ln"]).alias("metric_fmla_ln"),
                F.lit(spec["filter_expression"]).alias("filter_expression"),
                F.lit(spec["having_ln"]).alias("having_ln"),
                F.col(prefix + "_value").cast(DECIMAL_SQL).alias("val_amnt"),
                F.col(prefix + "_matched").cast("long").alias("matched_rows"),
                F.col(prefix + "_valid").cast("long").alias("valid_rows"),
                F.col(prefix + "_invalid").cast("long").alias("invalid_rows"),
                F.col(prefix + "_allnull").cast("long").alias("all_null_rows"),
            )
        )

    exploded = (
        aggregated
        .withColumn("_metric", F.explode(F.array(*structs)))
        .select(*grain_internal, "_metric.*")
    )
    key_json = key_json_expression(group["grain_columns"])
    return (
        exploded
        .withColumn("key_json_ln", key_json)
        .withColumn("key_hsh_cd", F.substring(F.sha2("key_json_ln", 256), 1, 32))
    )


def process_base_groups(plan, issues, process_run_id, metadata_ts, reference_period_value):
    """Calculates and writes the BASE variables group by group; returns the number of values written."""
    source_cache = {}
    failed_frames = []
    total_values = 0
    no_period_reported = set()
    unparseable_checked = set()

    try:
        for group in execution_groups(plan["base_specs"]):
            started = utc_now()
            source = period_source_frame(
                group["phy_tbl_nm"],
                group["physical_month"],
                group["per_column_nm"],
                group["per_fmt_cd"],
                source_cache,
                issues,
                unparseable_checked,
            )
            # Grain columns with their standard names; missing ones are NULL.
            source = prepare_grain_columns(source, group["grain_columns"])
            source_count = source.count()
            if source_count == 0:
                source_key = (
                    group["phy_tbl_nm"],
                    group["physical_month"],
                )
                if source_key not in no_period_reported:
                    issues.emit(
                        "CALC.INPUT.NO_PERIOD_ROWS",
                        source_table=group["phy_tbl_nm"],
                        group_id=group["grp_id"],
                        params={
                            "src_tbl_id": group["phy_tbl_nm"],
                            "per_cd": group["physical_month"],
                        },
                    )
                    no_period_reported.add(source_key)
                append_step(
                    process_run_id, "BASE_GROUP", "SKIPPED", 0,
                    group["grp_id"], "No rows for physical period.", started,
                )
                continue

            frame = build_base_group_frame(source, group).localCheckpoint(eager=True)
            try:
                classified = (
                    frame
                    .withColumn(
                        "reason",
                        F.when(F.col("invalid_rows") > 0, F.lit("INVALID_VALUE"))
                        .when(
                            (F.col("agg_fcn_cd") == "FIRST")
                            & (F.col("matched_rows") > 1),
                            F.lit("CARDINALITY"),
                        )
                        .when(F.col("matched_rows") == 0, F.lit("NO_DATA"))
                        .when(
                            (F.col("agg_fcn_cd") != "COUNT")
                            & (F.col("valid_rows") == 0),
                            F.lit("NO_VALID_VALUES"),
                        )
                        .otherwise(F.lit("OK")),
                    )
                ).persist(StorageLevel.MEMORY_AND_DISK)

                try:
                    issue_counts = list(
                        classified.where(F.col("reason") != "OK")
                        .groupBy(
                            "framework_cd", "datapoint_id", "var_id",
                            "var_cd", "src_tbl_id", "reason",
                        )
                        .count()
                        .toLocalIterator()
                    )
                    # An entity without rows for a variable cannot be told apart from a zero value (some
                    # source tables only store non-zero cells), so no issue is raised per entity.
                    # NO_CONTRIBUTING_ROWS is raised once, when the variable has no rows for any entity.
                    variables_with_rows = {
                        row["var_id"]
                        for row in classified.where(F.col("matched_rows") > 0)
                        .select("var_id").distinct().toLocalIterator()
                    }
                    for row in issue_counts:
                        reason = row["reason"]
                        count = int(row["count"])
                        local = row["var_cd"]
                        if reason == "NO_DATA" and row["var_id"] in variables_with_rows:
                            continue
                        common = dict(
                            framework=row["framework_cd"],
                            source_table=row["src_tbl_id"],
                            datapoint_id=row["datapoint_id"],
                            variable_id=row["var_id"],
                            group_id=group["grp_id"],
                            affected_count=count,
                        )
                        if reason == "NO_DATA":
                            issues.emit(
                                "CALC.AGG.NO_CONTRIBUTING_ROWS",
                                **common,
                                params={
                                    "datapoint_id": row["datapoint_id"],
                                    "var_cd": local,
                                    "src_tbl_id": row["src_tbl_id"],
                                    "per_cd": group["physical_month"],
                                    "afct_nr": count,
                                },
                            )
                        elif reason == "INVALID_VALUE":
                            issues.emit(
                                "CALC.INPUT.INVALID_VALUE",
                                **common,
                                params={
                                    "datapoint_id": row["datapoint_id"],
                                    "var_cd": local,
                                    "afct_nr": count,
                                    "affected_unit": "keys",
                                },
                            )
                        elif reason == "NO_VALID_VALUES":
                            issues.emit(
                                "CALC.INPUT.NO_VALID_VALUES",
                                **common,
                                params={
                                    "datapoint_id": row["datapoint_id"],
                                    "var_cd": local,
                                    "afct_nr": count,
                                    "affected_unit": "keys",
                                },
                            )
                        elif reason == "CARDINALITY":
                            issues.emit(
                                "CALC.CARDINALITY.MULTIPLE_ROWS",
                                **common,
                                params={
                                    "datapoint_id": row["datapoint_id"],
                                    "var_cd": local,
                                    "afct_nr": count,
                                    "affected_unit": "keys",
                                },
                            )

                    failed = (
                        classified.where(
                            F.col("reason").isin(
                                "INVALID_VALUE", "NO_VALID_VALUES", "CARDINALITY"
                            )
                        )
                        .select(
                            F.lit(process_run_id).alias("exec_id"),
                            "framework_cd", "datapoint_id", "var_id", "key_hsh_cd",
                            F.when(
                                F.col("reason") == "INVALID_VALUE",
                                F.lit("CALC.INPUT.INVALID_VALUE"),
                            ).when(
                                F.col("reason") == "NO_VALID_VALUES",
                                F.lit("CALC.INPUT.NO_VALID_VALUES"),
                            ).otherwise(
                                F.lit("CALC.CARDINALITY.MULTIPLE_ROWS")
                            ).alias("iss_cd"),
                            F.current_timestamp().alias("crtn_dt"),
                        )
                    )
                    if failed.limit(1).count():
                        append_to_table(failed, failed_key_table, "failed variable keys")

                    valid = classified.where(F.col("reason") == "OK")
                    if any(spec["having_ln"] for spec in group["specs"]):
                        # HAVING is evaluated after the aggregate and is expected suppression,
                        # therefore it never emits an issue.
                        having_cases = []
                        for spec in group["specs"]:
                            predicate = spec["having_ln"] or "TRUE"
                            having_cases.append(
                                (spec["var_id"], predicate)
                            )
                        condition = None
                        for variable_id, predicate in having_cases:
                            piece = (
                                (F.col("var_id") == variable_id)
                                & F.expr(predicate)
                            )
                            condition = piece if condition is None else (condition | piece)
                        valid = valid.where(condition)

                    # Keys published as 0 because all their rows had all metric columns empty.
                    # WARNING: the key does not fail.
                    all_null_counts = list(
                        valid
                        .where(F.col("agg_fcn_cd") != "COUNT")
                        .where(
                            (F.col("matched_rows") > 0)
                            & (F.col("all_null_rows") == F.col("matched_rows"))
                        )
                        .groupBy(
                            "framework_cd", "datapoint_id", "var_id",
                            "var_cd", "src_tbl_id",
                        )
                        .count()
                        .toLocalIterator()
                    )
                    for row in all_null_counts:
                        count = int(row["count"])
                        issues.emit(
                            "CALC.INPUT.ALL_NULL_METRIC",
                            framework=row["framework_cd"],
                            source_table=row["src_tbl_id"],
                            datapoint_id=row["datapoint_id"],
                            variable_id=row["var_id"],
                            group_id=group["grp_id"],
                            affected_count=count,
                            params={
                                "datapoint_id": row["datapoint_id"],
                                "var_cd": row["var_cd"],
                                "afct_nr": count,
                                "affected_unit": "keys",
                            },
                        )

                    values = valid.select(
                        F.lit(process_run_id).alias("exec_id"),
                        ref_dt_literal(reference_period_value).alias("ref_dt"),
                        "framework_cd", "datapoint_id", "var_id", "var_cd",
                        F.lit("BASE").alias("var_cl_typ"),
                        "src_tbl_id", "per_cd", "per_frq_cd", "grain_id",
                        "key_hsh_cd", "key_json_ln",
                        F.col("val_amnt").cast(DECIMAL_SQL).alias("val_amnt"),
                        F.when(
                            F.col("agg_fcn_cd") == "COUNT",
                            F.col("matched_rows"),
                        ).otherwise(F.col("valid_rows")).cast("long").alias("contributing_row_nr"),
                        F.lit(metadata_ts).cast("timestamp").alias("param_snpsh_dt"),
                        F.current_timestamp().alias("crtn_dt"),
                    )
                    row_count = values.count()
                    if row_count:
                        append_to_table(values, result_variable_table, "BASE variable values")
                    total_values += row_count
                    append_step(
                        process_run_id, "BASE_GROUP", "SUCCESS", row_count,
                        group["grp_id"], "", started,
                    )
                finally:
                    classified.unpersist()
            finally:
                frame.unpersist()
    finally:
        for frame in source_cache.values():
            frame.unpersist()

    return total_values


## 7. Derived variables


In [ ]:
def build_dependency_mapping_frame(derived_specs):
    """DERIVED variable to the BASE variables its formula uses; None when there are none."""
    rows = []
    for spec in derived_specs:
        for local, source_variable_id in spec["local_to_internal"].items():
            if local not in spec["required_locals"]:
                continue
            rows.append({
                "derived_variable_id": spec["var_id"],
                "framework_cd": spec["framework_cd"],
                "datapoint_id": spec["datapoint_id"],
                "var_cd": local,
                "source_variable_id": source_variable_id,
                "expected_dependency_count": len(spec["required_locals"]),
            })
    if not rows:
        return None
    return spark.createDataFrame(rows)


def process_derived_groups(
    plan, issues, process_run_id, metadata_ts, reference_period_value
):
    """Evaluates the formulas of the Calculated datapoints and writes the DERIVED values."""
    if not plan["derived_specs"]:
        return 0

    dependency_map = build_dependency_mapping_frame(plan["derived_specs"])
    base_values = (
        spark.table(result_variable_table)
        .where(F.col("exec_id") == process_run_id)
        .where(F.col("ref_dt") == ref_dt_literal(reference_period_value))
    )

    dep_values = (
        dependency_map.alias("m")
        .join(
            base_values.alias("v"),
            F.col("m.source_variable_id") == F.col("v.var_id"),
            "inner",
        )
        .select(
            "m.derived_variable_id", "m.framework_cd", "m.datapoint_id",
            "m.var_cd", "m.expected_dependency_count",
            "v.per_cd", "v.per_frq_cd", "v.grain_id",
            "v.key_hsh_cd", "v.key_json_ln", "v.val_amnt", "v.contributing_row_nr",
        )
    ).persist(StorageLevel.MEMORY_AND_DISK)

    failed_source = (
        spark.table(failed_key_table)
        .where(F.col("exec_id") == process_run_id)
        .alias("f")
        .join(
            dependency_map.alias("m"),
            F.col("f.var_id") == F.col("m.source_variable_id"),
            "inner",
        )
        .select(
            "m.derived_variable_id",
            "f.key_hsh_cd",
        )
        .dropDuplicates()
    ).persist(StorageLevel.MEMORY_AND_DISK)

    total_rows = 0
    try:
        dep_values.count()
        failed_source.count()

        group_keys = [
            "derived_variable_id", "framework_cd", "datapoint_id",
            "expected_dependency_count",
            "per_cd", "per_frq_cd", "grain_id", "key_hsh_cd", "key_json_ln",
        ]
        wide = (
            dep_values.groupBy(*group_keys)
            .pivot("var_cd", [f"V{i}" for i in range(1, MAX_VARIABLES_PER_DATAPOINT + 1)])
            .agg(F.first("val_amnt"))
        )
        contributing = (
            dep_values.groupBy(*group_keys)
            .agg(
                F.countDistinct("var_cd").cast("long").alias("present_dependency_count"),
                F.sum("contributing_row_nr").cast("long").alias("contributing_row_nr"),
            )
        )
        # A variable without a value for a key counts as 0, as in the previous solution.
        # Keys with a FAILED variable (non-numeric value, several rows per key...) are
        # still excluded through failed_source and are not published.
        variable_columns = [f"V{i}" for i in range(1, MAX_VARIABLES_PER_DATAPOINT + 1)]
        wide = wide.join(contributing, group_keys, "inner")
        for column in variable_columns:
            wide = wide.withColumn(
                column, F.coalesce(F.col(column), F.lit(0).cast(DECIMAL_SQL))
            )
        wide = (
            wide.join(failed_source, ["derived_variable_id", "key_hsh_cd"], "left_anti")
            .persist(StorageLevel.MEMORY_AND_DISK)
        )

        try:
            wide.count()
            specs_by_id = {spec["var_id"]: spec for spec in plan["derived_specs"]}

            for block_index, start in enumerate(
                range(0, len(plan["derived_specs"]), int(max_variables_per_group)),
                1,
            ):
                specs = plan["derived_specs"][
                    start:start + int(max_variables_per_group)
                ]
                ids = [spec["var_id"] for spec in specs]
                block = wide.where(F.col("derived_variable_id").isin(ids))

                value_branches = []
                div0_branches = []
                for spec in specs:
                    value_sql, div0_sql = arithmetic_sql(spec["fmla_ln"])
                    value_branches.append(
                        f"WHEN derived_variable_id = {sql_literal(spec['var_id'])} "
                        f"THEN {value_sql}"
                    )
                    div0_branches.append(
                        f"WHEN derived_variable_id = {sql_literal(spec['var_id'])} "
                        f"THEN ({div0_sql})"
                    )
                value_case_sql = "CASE " + " ".join(value_branches) + " ELSE NULL END"
                div0_case_sql = "CASE " + " ".join(div0_branches) + " ELSE FALSE END"

                evaluated = (
                    block
                    # A missing dependency is not an error (it counts as 0, see above).
                    .withColumn("missing_dependency", F.lit(False))
                    .withColumn("division_by_zero", F.expr(div0_case_sql))
                    .withColumn("_raw_result", F.expr(value_case_sql))
                    .withColumn(
                        "val_amnt",
                        F.expr(f"TRY_CAST(_raw_result AS {DECIMAL_SQL})"),
                    )
                    .withColumn(
                        "result_invalid",
                        (~F.col("missing_dependency"))
                        & (~F.col("division_by_zero"))
                        & F.col("val_amnt").isNull(),
                    )
                    .persist(StorageLevel.MEMORY_AND_DISK)
                )

                try:
                    # Materialize inside the block so arithmetic exceptions are isolated.
                    evaluated.count()

                    issue_counts = list(
                        evaluated.withColumn(
                            "reason",
                            F.when(F.col("missing_dependency"), F.lit("MISSING"))
                            .when(F.col("division_by_zero"), F.lit("DIV0"))
                            .when(F.col("result_invalid"), F.lit("INVALID"))
                            .otherwise(F.lit("OK")),
                        )
                        .where(F.col("reason") != "OK")
                        .groupBy(
                            "derived_variable_id", "framework_cd",
                            "datapoint_id", "reason",
                        )
                        .count()
                        .toLocalIterator()
                    )

                    for row in issue_counts:
                        spec = specs_by_id[row["derived_variable_id"]]
                        affected = int(row["count"])
                        common = dict(
                            framework=row["framework_cd"],
                            datapoint_id=row["datapoint_id"],
                            variable_id=row["derived_variable_id"],
                            group_id=f"DERIVED_B{block_index:03d}",
                            affected_count=affected,
                        )
                        if row["reason"] == "MISSING":
                            issues.emit(
                                "CALC.DERIVED.MISSING_DEPENDENCY",
                                **common,
                                params={
                                    "datapoint_id": row["datapoint_id"],
                                    "var_cd": "RESULT",
                                    "dependencies": ",".join(spec["required_locals"]),
                                    "afct_nr": affected,
                                },
                            )
                        elif row["reason"] == "DIV0":
                            issues.emit(
                                "CALC.DERIVED.DIVISION_BY_ZERO",
                                **common,
                                params={
                                    "datapoint_id": row["datapoint_id"],
                                    "var_cd": "RESULT",
                                    "afct_nr": affected,
                                },
                            )
                        else:
                            issues.emit(
                                "CALC.DERIVED.RESULT_INVALID",
                                **common,
                                params={
                                    "datapoint_id": row["datapoint_id"],
                                    "var_cd": "RESULT",
                                    "detail": "result is null or outside DECIMAL(38,10)",
                                    "afct_nr": affected,
                                },
                            )

                    failed = (
                        evaluated.withColumn(
                            "iss_cd",
                            F.when(
                                F.col("missing_dependency"),
                                F.lit("CALC.DERIVED.MISSING_DEPENDENCY"),
                            ).when(
                                F.col("division_by_zero"),
                                F.lit("CALC.DERIVED.DIVISION_BY_ZERO"),
                            ).when(
                                F.col("result_invalid"),
                                F.lit("CALC.DERIVED.RESULT_INVALID"),
                            ),
                        )
                        .where(F.col("iss_cd").isNotNull())
                        .select(
                            F.lit(process_run_id).alias("exec_id"),
                            "framework_cd", "datapoint_id",
                            F.col("derived_variable_id").alias("var_id"),
                            "key_hsh_cd", "iss_cd",
                            F.current_timestamp().alias("crtn_dt"),
                        )
                    )
                    if failed.limit(1).count():
                        append_to_table(failed, failed_key_table, "failed derived keys")

                    valid = evaluated.where(
                        (~F.col("missing_dependency"))
                        & (~F.col("division_by_zero"))
                        & (~F.col("result_invalid"))
                    ).select(
                        F.lit(process_run_id).alias("exec_id"),
                        ref_dt_literal(reference_period_value).alias("ref_dt"),
                        "framework_cd", "datapoint_id",
                        F.col("derived_variable_id").alias("var_id"),
                        F.lit("RESULT").alias("var_cd"),
                        F.lit("DERIVED").alias("var_cl_typ"),
                        F.lit("").alias("src_tbl_id"),
                        "per_cd", "per_frq_cd", "grain_id",
                        "key_hsh_cd", "key_json_ln",
                        F.col("val_amnt").cast(DECIMAL_SQL).alias("val_amnt"),
                        F.col("contributing_row_nr").cast("long").alias("contributing_row_nr"),
                        F.lit(metadata_ts).cast("timestamp").alias("param_snpsh_dt"),
                        F.current_timestamp().alias("crtn_dt"),
                    )
                    count = valid.count()
                    if count:
                        append_to_table(valid, result_variable_table, "DERIVED variable values")
                    total_rows += count
                    append_step(
                        process_run_id, "DERIVED_GROUP", "SUCCESS", count,
                        f"DERIVED_B{block_index:03d}", "", utc_now(),
                    )

                except Exception as exc:
                    # An already classified technical failure (e.g. a write failure) is not an
                    # arithmetic error: it is propagated so that the run records it.
                    if isinstance(exc, ClassifiedFailure):
                        raise
                    # One derived block failing technically must not be confused with a
                    # business arithmetic condition. Persist an unclassified runtime issue
                    # and continue with later blocks only when the Spark session remains usable.
                    for spec in specs:
                        issues.emit(
                            "CALC.DERIVED.ARITHMETIC_ERROR",
                            framework=spec["framework_cd"],
                            datapoint_id=spec["datapoint_id"],
                            variable_id=spec["var_id"],
                            group_id=f"DERIVED_B{block_index:03d}",
                            params={
                                "datapoint_id": spec["datapoint_id"],
                                "var_cd": "RESULT",
                                "detail": str(exc)[:1000],
                            },
                            raw_error_class=type(exc).__name__,
                            raw_message=str(exc),
                        )
                    append_step(
                        process_run_id, "DERIVED_GROUP", "ERROR", 0,
                        f"DERIVED_B{block_index:03d}", str(exc), utc_now(),
                    )
                finally:
                    evaluated.unpersist()
        finally:
            wide.unpersist()
    finally:
        dep_values.unpersist()
        failed_source.unpersist()

    return total_rows


## 8. Final datapoints and lineage


In [ ]:
def build_mapping_frame(mapping_rows):
    """DataFrame of the variable to datapoint mapping."""
    return spark.createDataFrame(mapping_rows)


def require_unique_mapping(frame, columns, table_name):
    # Unreadable reference table or with duplicates -> TECH.SOURCE.READ_FAILED.
    """Fails with TECH.SOURCE.READ_FAILED if a reference table cannot be read or has duplicated keys."""
    try:
        duplicate = frame.groupBy(*columns).count().where(F.col("count") > 1).take(1)
    except Exception as exc:
        raise ClassifiedFailure(
            "TECH.SOURCE.READ_FAILED",
            f"Unable to read source {table_name}: {technical_detail(exc)}",
            params={"src_tbl_id": table_name, "detail": technical_detail(exc)},
            source_table=table_name,
        ) from exc
    if duplicate:
        detail = f"must be unique on ({', '.join(columns)}); duplicate: {duplicate[0].asDict()}"
        raise ClassifiedFailure(
            "TECH.SOURCE.READ_FAILED",
            f"Unable to read source {table_name}: {detail}",
            params={"src_tbl_id": table_name, "detail": detail},
            source_table=table_name,
        )


def json_key(*names):
    """First of the given names present in the grouping key (key_json_ln)."""
    return F.coalesce(*[F.get_json_object(F.col("key_json_ln"), f"$.{name}") for name in names])


def normalize_string(column):
    """Column as trimmed text."""
    return F.trim(column.cast("string"))


# Consolidation rules: view, perimeter type, rule perimeter, and the framework and
# template pattern (SQL LIKE) the rule is limited to; empty = every framework or template.
# The F 17 branch of FINREP applies the accounting perimeter within the prudential view;
# ENTIND is the individual view, without hierarchy. Used by the calculation and by the
# datapoint lineage query.
CONSOLIDATION_RULES = (
    ("PRUD",   "PRUDENCIAL",     "P",  "",       ""),
    ("PRUD",   "CONTABILISTICO", "C",  "FINREP", "17.%"),
    ("CONT",   "CONTABILISTICO", "C",  "",       ""),
    ("AGREG",  "CONTABILISTICO", "C",  "",       ""),
    ("ENTIND", "CONTABILISTICO", None, "",       ""),
)


def build_enriched_aggregated_rows(contributions):
    # Reference tables with an optional explicit schema (reference_tables_schema).
    """Aggregated contributions consolidated by view and entity with the reference tables."""
    hierarchy_table = reference_table_name(entity_hierarchy_table)
    country_table = reference_table_name(ref_template_country_table)
    type_table = reference_table_name(ref_source_datapoint_type_table)
    map_type_table = reference_table_name(ref_template_perimeter_table)
    hierarchy = read_table_or_fail(hierarchy_table)
    country_map = read_table_or_fail(country_table)
    type_map = read_table_or_fail(type_table)
    map_type_map = read_table_or_fail(map_type_table)

    def normalize_period(column):
        """First six digits of a period column (YYYYMM)."""
        return F.substring(F.regexp_replace(normalize_string(column), "[^0-9]", ""), 1, 6)

    hierarchy_keys = hierarchy.select(
        normalize_period(F.col("ref_dt")).alias("hierarchy_period"),
        normalize_string(F.col("ent_id")).alias("hierarchy_entity"),
        normalize_string(F.col("ent_perim_cd")).alias("hierarchy_perimeter"),
        normalize_string(F.col("ent_lvl_2_id")).alias("cdentniv2"),
        normalize_string(F.col("ent_lvl_3_id")).alias("cdentniv3"),
    )
    # The tables by template are keyed by framework and template (as in the Catalog).
    country_keys = country_map.where(F.col("ctry_flg") == 1).select(
        F.upper(normalize_string(F.col("framework_cd"))).alias("country_framework_cd"),
        normalize_string(F.col("tmplt_cd")).alias("country_tmplt_cd"),
    )
    type_keys = type_map.select(
        # Case-insensitive join on the source table name, as in the previous solution.
        F.upper(normalize_string(F.col("src_tbl_id"))).alias("mapped_src_tbl_id"),
        normalize_string(F.col("datapoint_typ")).alias("datapoint_typ"),
    )
    # Perimeter type per template, used only by the ENTIND view (NULL if not found).
    map_type_keys = map_type_map.select(
        F.upper(normalize_string(F.col("framework_cd"))).alias("perimeter_framework_cd"),
        normalize_string(F.col("tmplt_cd")).alias("perimeter_tmplt_cd"),
        normalize_string(F.col("csl_perim_typ")).alias("mapped_csl_perim_typ"),
    )

    require_unique_mapping(hierarchy_keys, ["hierarchy_period", "hierarchy_entity", "hierarchy_perimeter"], hierarchy_table)
    require_unique_mapping(country_keys, ["country_framework_cd", "country_tmplt_cd"], f"{country_table} (ctry_flg=1)")
    require_unique_mapping(type_keys, ["mapped_src_tbl_id"], type_table)
    require_unique_mapping(map_type_keys, ["perimeter_framework_cd", "perimeter_tmplt_cd"], map_type_table)

    # The perimeter (P/C) and the template filter are attributes of the consolidation
    # RULE (a row of `views` below), not of the source data.
    prepared = (
        contributions
        .withColumn("source_entity", normalize_string(json_key("ENTIDADE", "ENTITY", "CDENTIDADE")))
        .withColumn("source_cdwa", json_key("CDWA", "cdwa"))
        .withColumn("src_tbl_id", F.upper(normalize_string(F.col("src_tbl_id"))))
        # Join keys with the reference tables by template (not stored).
        .withColumn("_framework_key", F.upper(normalize_string(F.col("framework_cd"))))
        .withColumn("_template_key", normalize_string(F.col("tmplt_cd")))
        .withColumn("source_period", normalize_period(F.col("ref_dt")))
        .join(type_keys, F.col("src_tbl_id") == F.col("mapped_src_tbl_id"), "left")
        .join(
            country_keys,
            (F.col("_framework_key") == F.col("country_framework_cd"))
            & (F.col("_template_key") == F.col("country_tmplt_cd")),
            "left",
        )
        .join(
            map_type_keys,
            (F.col("_framework_key") == F.col("perimeter_framework_cd"))
            & (F.col("_template_key") == F.col("perimeter_tmplt_cd")),
            "left",
        )
    )

    views_schema = T.StructType([
        T.StructField("csl_view_cd", T.StringType(), False),
        T.StructField("csl_perim_typ", T.StringType(), False),
        T.StructField("rule_perimeter", T.StringType(), True),
        T.StructField("framework_filter", T.StringType(), False),
        T.StructField("template_filter", T.StringType(), False),
    ])
    views = spark.createDataFrame(list(CONSOLIDATION_RULES), views_schema)

    enriched = (
        prepared.crossJoin(views)
        .where(
            ((F.col("framework_filter") == "") | (F.col("_framework_key") == F.col("framework_filter")))
            & ((F.col("template_filter") == "") | F.expr("_template_key LIKE template_filter"))
        )
        .join(
            hierarchy_keys,
            (F.col("source_period") == F.col("hierarchy_period"))
            & (F.col("source_entity") == F.col("hierarchy_entity"))
            & (F.col("rule_perimeter") == F.col("hierarchy_perimeter")),
            "left",
        )
        .where((F.col("csl_view_cd") != "AGREG") | (F.col("cdentniv2") == "F0005"))
        # ENTIND is only published for reported entities.
        .where((F.col("csl_view_cd") != "ENTIND") | F.col("source_entity").isNotNull())
        .withColumn(
            "csl_ent_cd",
            # The prefix depends on the perimeter of the RULE, not of the view: the F 17 branch
            # (PRUD view, perimeter C) uses 'c', as in the previous solution.
            F.when(F.col("csl_view_cd") == "ENTIND", F.col("source_entity"))
            .when(F.col("csl_view_cd") == "AGREG", F.col("cdentniv2"))
            # Equivalent to Oracle's NVL(CDENTNIV3, 'p'||ENTIDADE), where 'p'||NULL = 'p'.
            .when(
                F.col("rule_perimeter") == "P",
                F.coalesce(F.col("cdentniv3"), F.concat(F.lit("p"), F.coalesce(F.col("source_entity"), F.lit("")))),
            )
            .otherwise(F.coalesce(F.col("cdentniv3"), F.concat(F.lit("c"), F.coalesce(F.col("source_entity"), F.lit(""))))),
        )
        # ENTIND does not keep the country (all countries are added up) and takes its
        # perimeter type from the reference table per template. Other views keep the rule's.
        .withColumn(
            "ctry_cd",
            F.when(
                (F.col("csl_view_cd") != "ENTIND") & F.col("country_tmplt_cd").isNotNull(),
                F.col("source_cdwa"),
            ),
        )
        .withColumn(
            "csl_perim_typ",
            F.when(F.col("csl_view_cd") == "ENTIND", F.col("mapped_csl_perim_typ")).otherwise(F.col("csl_perim_typ")),
        )
    )
    physical_keys = [
        "exec_id", "ref_dt", "per_cd", "per_frq_cd",
        "framework_cd", "tmplt_cd", "row_cd", "column_cd", "datapoint_id",
        "param_snpsh_dt", "csl_ent_cd",
        "datapoint_typ", "csl_perim_typ", "csl_view_cd", "ctry_cd",
    ]
    return (
        enriched.groupBy(*physical_keys)
        .agg(
            F.sum("contribution_val_amnt").cast(DECIMAL_SQL).alias("val_amnt"),
            F.sum("contributing_row_nr").cast("long").alias("contributing_row_nr"),
        )
        .withColumn("grain_id", F.lit("ENRICHED_AGGREGATED"))
        # The perimeter type is part of the key: in F 17 the PRUD view produces a PRUDENCIAL
        # row and a CONTABILISTICO row with different key_hsh_cd.
        .withColumn("key_json_ln", F.to_json(F.struct(
            F.col("csl_view_cd").alias("csl_view_cd"),
            F.col("csl_perim_typ").alias("csl_perim_typ"),
            F.col("csl_ent_cd").alias("csl_ent_cd"),
            F.col("ctry_cd").alias("ctry_cd"),
            F.col("datapoint_typ").alias("datapoint_typ"),
        ), {"ignoreNullFields": "false"}))
        .withColumn("key_hsh_cd", F.substring(F.sha2("key_json_ln", 256), 1, 32))
        .withColumn("crtn_dt", F.current_timestamp())
        .select(
            "exec_id", "ref_dt", "per_cd", "per_frq_cd",
            "framework_cd", "tmplt_cd", "row_cd", "column_cd", "datapoint_id",
            "grain_id", "key_hsh_cd", "key_json_ln", "param_snpsh_dt",
            "val_amnt", "contributing_row_nr", "csl_ent_cd",
            "datapoint_typ", "csl_perim_typ", "csl_view_cd", "ctry_cd", "crtn_dt",
        )
    )


def process_final_datapoints(
    plan, issues, process_run_id, metadata_ts, reference_period_value
):
    """Builds and writes the datapoint values of the run; returns the number of rows written."""
    if not plan["mapping_rows"]:
        return 0
    mapping = build_mapping_frame(plan["mapping_rows"])
    values = (
        spark.table(result_variable_table)
        .where(F.col("exec_id") == process_run_id)
        .where(F.col("ref_dt") == ref_dt_literal(reference_period_value))
    )

    contributions = (
        values.alias("v")
        .join(mapping.alias("m"), "var_id", "inner")
        .select(
            "v.exec_id", "v.ref_dt", "v.per_cd", "v.per_frq_cd",
            "v.framework_cd", "m.tmplt_cd", "m.row_cd", "m.column_cd",
            "m.datapoint_id", "var_id", "v.grain_id", "m.metadata_type",
            "m.src_tbl_id", "v.key_hsh_cd", "v.key_json_ln",
            # The contribution carries the sign of the mapping (+1, or -1 in a Rollup). It is
            # negated instead of multiplied: DECIMAL(38,10) * integer would reduce the scale.
            F.when(F.col("m.sign") < 0, -F.col("v.val_amnt").cast(DECIMAL_SQL))
            .otherwise(F.col("v.val_amnt").cast(DECIMAL_SQL))
            .cast(DECIMAL_SQL).alias("contribution_val_amnt"),
            F.col("v.contributing_row_nr").cast("long").alias("contributing_row_nr"),
            F.lit(metadata_ts).cast("timestamp").alias("param_snpsh_dt"),
        )
    ).persist(StorageLevel.MEMORY_AND_DISK)

    try:
        contributions.count()
        failed = (
            spark.table(failed_key_table).where(F.col("exec_id") == process_run_id)
            .alias("f").join(mapping.alias("m"), "var_id", "inner")
            .select(F.col("m.datapoint_id").alias("datapoint_id"), F.col("f.key_hsh_cd").alias("key_hsh_cd"))
            .dropDuplicates()
        )
        clean_contributions = contributions.join(failed, ["datapoint_id", "key_hsh_cd"], "left_anti").persist(StorageLevel.MEMORY_AND_DISK)
        try:
            clean_contributions.count()
            calculated = (
                clean_contributions.where(F.col("metadata_type") != "AGGREGATED")
                .groupBy(
                    "exec_id", "ref_dt", "per_cd", "per_frq_cd",
                    "framework_cd", "tmplt_cd", "row_cd", "column_cd", "datapoint_id",
                    "grain_id", "key_hsh_cd", "key_json_ln", "param_snpsh_dt",
                )
                .agg(F.sum("contribution_val_amnt").cast(DECIMAL_SQL).alias("val_amnt"), F.sum("contributing_row_nr").cast("long").alias("contributing_row_nr"))
                # The entity is taken from the calculation key (IDENT first), as in the previous
                # solution, instead of NULL.
                .withColumn(
                    "csl_ent_cd",
                    normalize_string(json_key("IDENT", "ENTIDADE", "ENTIDADEAGG", "ENTITY", "CDENTIDADE")),
                )
                .withColumn("datapoint_typ", F.lit(None).cast("string"))
                .withColumn("csl_perim_typ", F.lit(None).cast("string"))
                .withColumn("csl_view_cd", F.lit(None).cast("string"))
                .withColumn("ctry_cd", F.lit(None).cast("string"))
                .withColumn("crtn_dt", F.current_timestamp())
            )
            aggregate_contributions = clean_contributions.where(F.col("metadata_type") == "AGGREGATED")
            if aggregate_contributions.limit(1).count():
                final = calculated.unionByName(
                    build_enriched_aggregated_rows(aggregate_contributions)
                ).persist(StorageLevel.MEMORY_AND_DISK)
            else:
                final = calculated.persist(StorageLevel.MEMORY_AND_DISK)
            try:
                final_count = final.count()
                if final_count:
                    append_to_table(final, result_datapoint_table, "datapoint values")
                return final_count
            finally:
                final.unpersist()
        finally:
            clean_contributions.unpersist()
    finally:
        contributions.unpersist()


def persist_variable_lineage(
    plan, process_run_id, metadata_ts, reference_period_value
):
    """Variable lineage: one row per BASE variable with the query that calculates it on
    the source table (src_sql_ln). The values per key are in result_var_val."""
    rows = [
        {
            "exec_id": process_run_id,
            "framework_cd": spec["framework_cd"],
            "datapoint_id": spec["datapoint_id"],
            "var_id": spec["var_id"],
            "var_cd": spec["var_cd"],
            "src_tbl_id": spec["src_tbl_id"],
            "agg_fcn_cd": spec["agg_fcn_cd"],
            "metric_fmla_ln": spec["metric_fmla_ln"],
            "having_ln": spec["having_ln"],
            # Complete SELECT that reproduces the value of the variable.
            "src_sql_ln": spec["source_sql"],
            "grain_id": spec["grain_id"],
            "per_cd": spec["per_cd"],
        }
        for spec in plan["base_specs"]
    ]
    if not rows:
        return 0
    schema = T.StructType([T.StructField(name, T.StringType()) for name in rows[0]])
    lineage = (
        spark.createDataFrame(rows, schema)
        .withColumn("ref_dt", ref_dt_literal(reference_period_value))
        .withColumn("param_snpsh_dt", F.lit(metadata_ts).cast("timestamp"))
        .withColumn("crtn_dt", F.current_timestamp())
        .select(
            "exec_id", "ref_dt", "framework_cd", "datapoint_id", "var_id", "var_cd",
            "src_tbl_id", "agg_fcn_cd", "metric_fmla_ln", "having_ln", "src_sql_ln",
            "grain_id", "per_cd", "param_snpsh_dt", "crtn_dt",
        )
    )
    append_to_table(lineage, lineage_variable_table, "variable lineage")
    return len(rows)


# ---------------------------------------------------------------------------
# Datapoint lineage: query that builds the rows of each datapoint in
# result_datapoint_val from result_var_val. It reproduces process_final_datapoints()
# and, for Aggregated datapoints, build_enriched_aggregated_rows(): keep them aligned.
# ---------------------------------------------------------------------------
def _sql_timestamp(value):
    """Reference period as a SQL TIMESTAMP literal."""
    moment = reference_date(value)
    return f"TIMESTAMP'{moment:%Y-%m-%d %H:%M:%S}'" if moment else "CAST(NULL AS TIMESTAMP)"


def _like_matches(value, pattern):
    """True if the value matches a SQL LIKE pattern."""
    regex = "^" + re.escape(pattern).replace("%", ".*").replace("_", ".") + "$"
    return re.match(regex, value) is not None


def _rule_applies(framework_key, template_key, framework_filter, template_filter):
    """True if a consolidation rule applies to the framework and template (same test as
    the calculation)."""
    framework_ok = not framework_filter or framework_key == framework_filter
    template_ok = not template_filter or _like_matches(template_key, template_filter)
    return framework_ok and template_ok


# Value of an active flag (ctry_flg) in the reference tables. It is written from a constant
# so that the lineage query keeps no literal number in its filters.
ACTIVE_FLAG_SQL = "1"


def indent_sql(text, spaces):
    """Text with its continuation lines indented, to place it inside a query."""
    return text.replace("\n", "\n" + " " * spaces)


def _variables_condition(item, process_run_id, reference_period_value):
    """Filter of the datapoint lineage query: run, period, variables and no failed keys."""
    var_list = ", ".join(sql_literal(v["var_id"]) for v in item["variables"])
    return "\n".join([
        f"v.exec_id = {sql_literal(process_run_id)}",
        f"AND v.ref_dt = {_sql_timestamp(reference_period_value)}",
        f"AND v.var_id IN ({var_list})",
        "-- keys with a failed variable are not published",
        "AND NOT EXISTS (",
        "    SELECT 1",
        f"    FROM {failed_key_table} AS f",
        "    WHERE",
        "        f.exec_id = v.exec_id",
        "        AND f.key_hsh_cd = v.key_hsh_cd",
        f"        AND f.var_id IN ({var_list})",
        ")",
    ])


def _signed_value_sql(item):
    """CASE expression with the value of each variable and its sign."""
    branches = " ".join(
        f"WHEN {sql_literal(v['var_id'])} THEN {'-' if v['sign'] < 0 else ''}v.val_amnt"
        for v in item["variables"]
    )
    return f"CASE v.var_id {branches} END"


def _source_table_sql(item):
    """CASE expression with the source table of each variable."""
    branches = " ".join(
        f"WHEN {sql_literal(v['var_id'])} THEN {sql_literal(v['src_tbl_id'])}"
        for v in item["variables"]
    )
    return f"CASE v.var_id {branches} END"


def _default_lineage_sql(item, process_run_id, reference_period_value):
    """Lineage query of a Default: published in each entity and view with results for the
    framework."""
    return f"""
SELECT DISTINCT
    r.csl_ent_cd
    , r.csl_view_cd
    , r.csl_perim_typ
    , r.datapoint_typ
    , CAST(NULL AS STRING) AS ctry_cd
    , {sql_literal(item['fmla_ln'])} AS def_val_ln
FROM {result_datapoint_table} AS r
WHERE
    r.exec_id = {sql_literal(process_run_id)}
    AND r.ref_dt = {_sql_timestamp(reference_period_value)}
    AND r.framework_cd = {sql_literal(item['framework_cd'])}
""".strip()


def _formula_lineage_sql(item, condition):
    """Lineage query of a Calculated with a formula: a variable missing for a key counts
    as 0; a division by zero leaves the key unpublished."""
    value_sql, _ = arithmetic_sql(item["fmla_ln"])
    for local, var_id in sorted(item["formula_vars"].items(), key=lambda kv: -len(kv[0])):
        value_sql = value_sql.replace(
            quote_identifier(local),
            f"COALESCE(SUM(CASE WHEN v.var_id = {sql_literal(var_id)} THEN v.val_amnt END), "
            f"CAST(0 AS {DECIMAL_SQL}))",
        )
    return f"""
SELECT
    v.key_hsh_cd
    , v.key_json_ln
    , TRY_CAST({value_sql} AS DECIMAL(38, 10)) AS val_amnt
FROM {result_variable_table} AS v
WHERE
    {indent_sql(condition, 4)}
GROUP BY
    v.key_hsh_cd
    , v.key_json_ln
HAVING TRY_CAST({value_sql} AS DECIMAL(38, 10)) IS NOT NULL
""".strip()


def _signed_sum_lineage_sql(item, condition):
    """Lineage query of a single-variable Calculated or a Rollup of Calculated: signed sum
    per key."""
    return f"""
SELECT
    v.key_hsh_cd
    , v.key_json_ln
    , CAST(SUM({_signed_value_sql(item)}) AS DECIMAL(38, 10)) AS val_amnt
    , SUM(v.contributing_row_nr) AS contributing_row_nr
FROM {result_variable_table} AS v
WHERE
    {indent_sql(condition, 4)}
GROUP BY
    v.key_hsh_cd
    , v.key_json_ln
""".strip()


def _consolidated_lineage_sql(item, condition, reference_period_value):
    """Lineage query of an Aggregated or a Rollup of Aggregated: signed sum and
    consolidation by views."""
    framework_key = clean(item["framework_cd"]).upper()
    template_key = clean(item["tmplt_cd"])
    framework_literal = sql_literal(framework_key)
    template_literal = sql_literal(template_key)
    rules_sql = "\n            , ".join(
        f"({sql_literal(view)}, {sql_literal(perimeter_type)}, "
        f"{sql_literal(perimeter) if perimeter else 'CAST(NULL AS STRING)'})"
        for view, perimeter_type, perimeter, framework_filter, pattern in CONSOLIDATION_RULES
        if _rule_applies(framework_key, template_key, framework_filter, pattern)
    )
    period = sql_literal(re.sub(r"\D", "", str(reference_date(reference_period_value) or ""))[:6])
    hierarchy = reference_table_name(entity_hierarchy_table)
    country = reference_table_name(ref_template_country_table)
    types = reference_table_name(ref_source_datapoint_type_table)
    perimeter_map = reference_table_name(ref_template_perimeter_table)
    return f"""
WITH contributions AS (
    SELECT
        v.key_json_ln
        , v.contributing_row_nr
        , {_signed_value_sql(item)} AS contribution_val_amnt
        , {_source_table_sql(item)} AS src_tbl_id
    FROM {result_variable_table} AS v
    WHERE
        {indent_sql(condition, 8)}
)

, source_types AS (
    SELECT
        UPPER(TRIM(CAST(src_tbl_id AS STRING))) AS src_tbl_id
        , TRIM(CAST(datapoint_typ AS STRING)) AS datapoint_typ
    FROM {types}
)

, prepared AS (
    SELECT
        c.*
        , t.datapoint_typ
        , TRIM(CAST(COALESCE(
            GET_JSON_OBJECT(c.key_json_ln, '$.ENTIDADE')
            , GET_JSON_OBJECT(c.key_json_ln, '$.ENTITY')
            , GET_JSON_OBJECT(c.key_json_ln, '$.CDENTIDADE')
        ) AS STRING)) AS source_entity
        , COALESCE(
            GET_JSON_OBJECT(c.key_json_ln, '$.CDWA')
            , GET_JSON_OBJECT(c.key_json_ln, '$.cdwa')
        ) AS source_cdwa
        , (
            SELECT MAX(1)
            FROM {country} AS p
            WHERE
                p.ctry_flg = {ACTIVE_FLAG_SQL}
                AND UPPER(TRIM(CAST(p.framework_cd AS STRING))) = {framework_literal}
                AND TRIM(CAST(p.tmplt_cd AS STRING)) = {template_literal}
        ) AS country_flg
        , (
            SELECT MAX(TRIM(CAST(m.csl_perim_typ AS STRING)))
            FROM {perimeter_map} AS m
            WHERE
                UPPER(TRIM(CAST(m.framework_cd AS STRING))) = {framework_literal}
                AND TRIM(CAST(m.tmplt_cd AS STRING)) = {template_literal}
        ) AS mapped_csl_perim_typ
    FROM contributions AS c
    LEFT JOIN source_types AS t
        ON UPPER(TRIM(c.src_tbl_id)) = t.src_tbl_id
)

, rules AS (
    SELECT
        csl_view_cd
        , rule_perim_typ
        , rule_perimeter
    FROM
        VALUES {rules_sql} AS r (csl_view_cd, rule_perim_typ, rule_perimeter)
)

, hierarchy AS (
    SELECT
        TRIM(CAST(ent_id AS STRING)) AS entity
        , TRIM(CAST(ent_perim_cd AS STRING)) AS perimeter
        , TRIM(CAST(ent_lvl_2_id AS STRING)) AS cdentniv2
        , TRIM(CAST(ent_lvl_3_id AS STRING)) AS cdentniv3
    FROM {hierarchy}
    WHERE SUBSTRING(REGEXP_REPLACE(TRIM(CAST(ref_dt AS STRING)), '[^0-9]', ''), 1, 6) = {period}
)

, enriched AS (
    SELECT
        p.*
        , r.csl_view_cd
        , r.rule_perimeter
        , h.cdentniv2
        , h.cdentniv3
        , CASE
            WHEN r.csl_view_cd = 'ENTIND' THEN p.mapped_csl_perim_typ
            ELSE r.rule_perim_typ
        END AS csl_perim_typ
    FROM prepared AS p
    INNER JOIN rules AS r
        ON TRUE
    LEFT JOIN hierarchy AS h
        ON
            p.source_entity = h.entity
            AND r.rule_perimeter = h.perimeter
    WHERE
        (r.csl_view_cd <> 'AGREG' OR h.cdentniv2 = 'F0005')
        AND (r.csl_view_cd <> 'ENTIND' OR p.source_entity IS NOT NULL)
)

SELECT
    csl_view_cd
    , csl_perim_typ
    , CASE
        WHEN csl_view_cd = 'ENTIND' THEN source_entity
        WHEN csl_view_cd = 'AGREG' THEN cdentniv2
        WHEN rule_perimeter = 'P' THEN COALESCE(cdentniv3, CONCAT('p', COALESCE(source_entity, '')))
        ELSE COALESCE(cdentniv3, CONCAT('c', COALESCE(source_entity, '')))
    END AS csl_ent_cd
    , CASE WHEN csl_view_cd <> 'ENTIND' AND country_flg IS NOT NULL THEN source_cdwa END AS ctry_cd
    , datapoint_typ
    , CAST(SUM(contribution_val_amnt) AS DECIMAL(38, 10)) AS val_amnt
    , SUM(contributing_row_nr) AS contributing_row_nr
FROM enriched
GROUP BY 1, 2, 3, 4, 5
""".strip()


def datapoint_lineage_sql(item, process_run_id, reference_period_value):
    """Query that rebuilds the rows of a datapoint from result_var_val, consolidation included."""
    if item["datapoint_cl_typ"] == "DEFAULT":
        return _default_lineage_sql(item, process_run_id, reference_period_value)
    condition = _variables_condition(item, process_run_id, reference_period_value)
    if item["csl_flg"]:
        return _consolidated_lineage_sql(item, condition, reference_period_value)
    if item["formula_vars"]:
        return _formula_lineage_sql(item, condition)
    return _signed_sum_lineage_sql(item, condition)


def persist_datapoint_lineage(plan, process_run_id, metadata_ts, reference_period_value):
    """Datapoint lineage: one row per datapoint with its formula, the variables used with
    their sign and the query that builds its rows in result_datapoint_val (or in
    datapoint_def, for a Default)."""
    rows = []
    for item in plan.get("datapoint_lineage_items") or []:
        rows.append({
            "exec_id": process_run_id,
            "per_cd": item["per_cd"],
            "per_frq_cd": item["per_frq_cd"],
            "framework_cd": item["framework_cd"],
            "datapoint_id": item["datapoint_id"],
            "tmplt_cd": item["tmplt_cd"],
            "row_cd": item["row_cd"],
            "column_cd": item["column_cd"],
            "datapoint_cl_typ": item["datapoint_cl_typ"],
            "fmla_ln": item["fmla_ln"],
            "var_map_json_ln": json.dumps(
                [
                    {
                        "datapoint_id": v["datapoint_id"], "var_cd": v["var_cd"],
                        "var_id": v["var_id"], "sign": v["sign"],
                    }
                    for v in item["variables"]
                ],
                ensure_ascii=False,
            ),
            "datapoint_sql_ln": datapoint_lineage_sql(item, process_run_id, reference_period_value),
            "csl_flg": int(item["csl_flg"]),
        })
    if not rows:
        return 0
    schema = T.StructType(
        [T.StructField(name, T.StringType()) for name in rows[0] if name != "csl_flg"]
        + [T.StructField("csl_flg", T.IntegerType())]
    )
    ordered = [tuple(row[field.name] for field in schema.fields) for row in rows]
    lineage = (
        spark.createDataFrame(ordered, schema)
        .withColumn("ref_dt", ref_dt_literal(reference_period_value))
        .withColumn("param_snpsh_dt", F.lit(metadata_ts).cast("timestamp"))
        .withColumn("crtn_dt", F.current_timestamp())
        .select(
            "exec_id", "ref_dt", "per_cd", "per_frq_cd", "framework_cd", "datapoint_id",
            "tmplt_cd", "row_cd", "column_cd", "datapoint_cl_typ", "fmla_ln",
            "var_map_json_ln", "csl_flg", "datapoint_sql_ln", "param_snpsh_dt", "crtn_dt",
        )
    )
    append_to_table(lineage, lineage_datapoint_table, "datapoint lineage")
    return len(rows)


DEFAULT_ITEM_SCHEMA = T.StructType([
    T.StructField(name, T.StringType())
    for name in ("framework_cd", "datapoint_id", "tmplt_cd", "row_cd", "column_cd", "def_val_ln")
])
VALUE_TYPE_SCHEMA = T.StructType([
    T.StructField(name, T.StringType()) for name in ("framework_cd", "datapoint_id", "val_typ")
])
CONTEXT_COLUMNS = ("csl_ent_cd", "csl_view_cd", "csl_perim_typ", "datapoint_typ")


def process_defaults(plan, issues, process_run_id, metadata_ts, reference_period_value):
    """Publish each Default in every combination of entity and view (csl_ent_cd,
    csl_view_cd, csl_perim_typ, datapoint_typ) with results in this run and framework,
    without country (ctry_cd)."""
    items = plan.get("default_items") or []
    if not items:
        return 0
    defaults = spark.createDataFrame(
        [tuple(item[field.name] for field in DEFAULT_ITEM_SCHEMA.fields) for item in items],
        DEFAULT_ITEM_SCHEMA,
    )
    targets = (
        spark.table(result_datapoint_table)
        .where(F.col("exec_id") == process_run_id)
        .where(F.col("ref_dt") == ref_dt_literal(reference_period_value))
        .select("framework_cd", *CONTEXT_COLUMNS)
        .distinct()
    )
    with_targets = {row["framework_cd"] for row in targets.select("framework_cd").distinct().toLocalIterator()}
    for framework, default_count in sorted(Counter(item["framework_cd"] for item in items).items()):
        if framework not in with_targets:
            issues.emit(
                "CALC.DEFAULT.NO_TARGET_KEYS",
                framework=framework,
                affected_count=default_count,
                params={"framework_cd": framework, "default_count": default_count, "exec_id": process_run_id},
            )
    published = (
        defaults.join(targets, "framework_cd", "inner")
        .select(
            F.lit(process_run_id).alias("exec_id"),
            ref_dt_literal(reference_period_value).alias("ref_dt"),
            "framework_cd", "datapoint_id", "tmplt_cd", "row_cd", "column_cd",
            *CONTEXT_COLUMNS,
            F.lit(None).cast("string").alias("ctry_cd"),
            "def_val_ln",
            F.lit(metadata_ts).cast("timestamp").alias("param_snpsh_dt"),
            F.current_timestamp().alias("crtn_dt"),
        )
    )
    count = published.count()
    if count:
        append_to_table(published, datapoint_default_table, "default values")
    return count


REPORT_COLUMNS = (
    "exec_id", "ref_dt", "framework_cd", "datapoint_id", "tmplt_cd", "row_cd", "column_cd",
    "csl_ent_cd", "csl_view_cd", "csl_perim_typ", "datapoint_typ", "ctry_cd", "rpt_val_ln",
)


def publish_report(plan, process_run_id, reference_period_value, run_frameworks):
    """Publish the results and defaults of this run in the publication table, with the
    value formatted according to the value type. replaceWhere by period and frameworks
    of the run: the table always holds only the latest run of each period and framework."""
    value_types = (
        spark.createDataFrame(
            [tuple(row[field.name] for field in VALUE_TYPE_SCHEMA.fields) for row in plan.get("value_type_rows") or []],
            VALUE_TYPE_SCHEMA,
        )
        .dropDuplicates(["framework_cd", "datapoint_id"])
    )

    def with_type(frame):
        """Adds the value type of each datapoint (the default type when missing)."""
        return frame.join(value_types, ["framework_cd", "datapoint_id"], "left").withColumn(
            "val_typ", F.coalesce(F.col("val_typ"), F.lit(DEFAULT_VALUE_TYPE))
        )

    def this_run(table_name):
        """Rows of the table for this run and reference period."""
        return (
            spark.table(table_name)
            .where(F.col("exec_id") == process_run_id)
            .where(F.col("ref_dt") == ref_dt_literal(reference_period_value))
        )

    calculated = with_type(this_run(result_datapoint_table)).withColumn(
        "rpt_val_ln", format_decimal_value(F.col("val_amnt"), F.col("val_typ"))
    )
    defaults = with_type(this_run(datapoint_default_table)).withColumn(
        "rpt_val_ln", format_default_value("def_val_ln", F.col("val_typ"))
    )
    report = (
        calculated.select(*REPORT_COLUMNS)
        .unionByName(defaults.select(*REPORT_COLUMNS))
        .withColumn("pbh_dt", F.current_timestamp())
    ).persist(StorageLevel.MEMORY_AND_DISK)
    try:
        count = report.count()
        predicate = (
            f"ref_dt = {_sql_timestamp(reference_period_value)} AND framework_cd IN ("
            + ", ".join(sql_literal(framework) for framework in run_frameworks)
            + ")"
        )
        replace_in_table(report, report_datapoint_table, predicate, "report datapoints")
        return count
    finally:
        report.unpersist()


## 9. Execute run


In [ ]:
# The run is recorded first and all the preparation (issue catalogue, parameterisation
# snapshot, validation gate, plan) runs inside the try block, so any failure leaves a
# row in the run table and, if the catalogue could be loaded, an issue with its code.
def run_processing():
    """Calculates and publishes one reference period; returns the run summary.

    Runs the whole Processing flow described in the cell comment. On failure the run is
    recorded as FAILED and the exception is raised again.
    """
    # Assigned here and read by source_config_for() (Source / grain contract cell).
    global SOURCE_CONFIG
    started_at = utc_now()
    requested_frameworks = split_csv(frameworks_csv)
    reference_period_value = clean(reference_period)
    frameworks = list(requested_frameworks)
    resolved_metadata_as_of = None
    issues = None
    # Readable identifier in UTC; the run_id parameter can still force it.
    process_run_id = clean(run_id) or build_run_id(
        requested_frameworks, reference_period, run_mode, started_at
    )
    print(f"exec_id: {process_run_id}")


    def run_row(status, warning_count=0, error_count=0, finished_at=None, message=""):
        """Row of the run table with the current state of the run."""
        return {
            "exec_id": process_run_id,
            "ref_dt": reference_period_value,
            "framework_list_ln": ",".join(frameworks),
            "param_snpsh_dt": resolved_metadata_as_of,
            "exec_mode_cd": clean(run_mode) or "full",
            "exec_stts": status,
            "warning_nr": warning_count,
            "err_nr": error_count,
            "start_dt": started_at,
            "end_dt": finished_at,
            "msg_ln": message,
            "req_id": clean(req_id),
        }


    upsert_run_execution(run_row("RUNNING"))

    active_step = "PREPARATION"
    try:
        active_step = "ISSUE_CATALOG"
        issue_catalog = load_issue_catalog()
        issues = ProcessingIssues(issue_catalog, process_run_id, None, reference_period_value)
        check_issue_catalog(issue_catalog)

        active_step = "PARAMETERS"
        try:
            reference_period_value = validate_reference_period(reference_period)
        except ValueError as exc:
            raise ClassifiedFailure(
                "TECH.PARAM.INVALID",
                str(exc),
                params={
                    "parameter": "reference_period",
                    "value": clean(reference_period),
                    "detail": "Use YYYYMM or YYYYMMDD.",
                },
            ) from exc
        issues.reference_period = reference_period_value

        active_step = "METADATA_SNAPSHOT"
        resolved_metadata_as_of = resolve_metadata_as_of(metadata_as_of)
        issues.metadata_ts = resolved_metadata_as_of
        SOURCE_CONFIG = load_source_config(
            scd_snapshot(source_config_table, resolved_metadata_as_of)
        )
        datapoints, filters, catalog = select_metadata_snapshot(
            resolved_metadata_as_of,
            requested_frameworks,
        )

        available_frameworks = sorted({
            clean(row["framework_cd"]).upper()
            for row in datapoints.select("framework_cd").distinct().toLocalIterator()
            if clean(row["framework_cd"])
        })
        frameworks = sorted(requested_frameworks or available_frameworks)
        if not frameworks:
            raise ClassifiedFailure(
                "META.SELECTION.EMPTY",
                "No frameworks were selected for Processing.",
            )
        upsert_run_execution(run_row("RUNNING"))

        active_step = "VALIDATION_GATE"
        require_validated_snapshot(frameworks, resolved_metadata_as_of, issues)

        active_step = "COMPILE_PLAN"
        try:
            plan = compile_plan(datapoints, filters, catalog, reference_period_value)
        except ClassifiedFailure:
            raise
        except Exception as exc:
            # The parameterisation is validated: if it still does not compile, an invariant is broken.
            raise ClassifiedFailure(
                "TECH.RUNTIME.INVARIANT_FAILED",
                technical_detail(exc),
                params={"stage": "COMPILE_PLAN", "detail": technical_detail(exc)},
            ) from exc

        active_step = "BASE"
        base_count = process_base_groups(
            plan,
            issues,
            process_run_id,
            resolved_metadata_as_of,
            reference_period_value,
        )

        active_step = "DERIVED"
        derived_count = process_derived_groups(
            plan,
            issues,
            process_run_id,
            resolved_metadata_as_of,
            reference_period_value,
        )

        active_step = "LINEAGE_VARIABLE"
        lineage_variable_count = persist_variable_lineage(
            plan,
            process_run_id,
            resolved_metadata_as_of,
            reference_period_value,
        )

        active_step = "DATAPOINT"
        datapoint_count = process_final_datapoints(
            plan,
            issues,
            process_run_id,
            resolved_metadata_as_of,
            reference_period_value,
        )

        # One row per datapoint with the query that builds its rows.
        active_step = "LINEAGE_DATAPOINT"
        lineage_datapoint_count = persist_datapoint_lineage(
            plan,
            process_run_id,
            resolved_metadata_as_of,
            reference_period_value,
        )

        # Defaults on the entities and views with results in this run.
        active_step = "DEFAULTS"
        step_started = utc_now()
        default_count = process_defaults(
            plan,
            issues,
            process_run_id,
            resolved_metadata_as_of,
            reference_period_value,
        )
        append_step(process_run_id, "DEFAULTS", "SUCCESS", default_count, "", "", step_started)

        active_step = "ISSUE_LOG"
        issues.persist()
        warning_count = sum(1 for row in issues.rows if row["sev_cd"] == "WARNING")
        error_count = sum(1 for row in issues.rows if row["sev_cd"] == "ERROR")

        # Publication at the end, once nothing in the calculation can fail. If it fails, the
        # run is set to FAILED and the publication table keeps the previous run.
        active_step = "REPORT"
        step_started = utc_now()
        report_count = publish_report(plan, process_run_id, reference_period_value, frameworks)
        append_step(process_run_id, "REPORT", "SUCCESS", report_count, "", "", step_started)

        if error_count:
            final_status = "COMPLETED_WITH_ERRORS"
        elif warning_count:
            final_status = "SUCCESS_WITH_WARNINGS"
        else:
            final_status = "SUCCESS"

        upsert_run_execution(run_row(final_status, warning_count, error_count, utc_now()))

        summary = {
            "exec_stts": final_status,
            "exec_id": process_run_id,
            "req_id": clean(req_id),
            "ref_dt": str(reference_date(reference_period_value)),
            "framework_list_ln": frameworks,
            "param_snpsh_dt": str(resolved_metadata_as_of),
            "base_variable_rows": base_count,
            "derived_variable_rows": derived_count,
            "variable_lineage_rows": lineage_variable_count,
            "datapoint_lineage_rows": lineage_datapoint_count,
            "datapoint_rows": datapoint_count,
            "default_rows": default_count,
            "report_rows": report_count,
            "warnings": warning_count,
            "errors": error_count,
        }
        print(json.dumps(summary, indent=2))

        display(
            spark.table(report_datapoint_table)
            .where(F.col("exec_id") == process_run_id)
            .orderBy("framework_cd", "datapoint_id", "csl_view_cd", "csl_ent_cd")
        )
        display(
            spark.table(issue_log_table)
            .where(F.col("exec_id") == process_run_id)
            .orderBy("sev_cd", "iss_cd", "datapoint_id")
        )
        return summary

    except Exception as exc:
        # Technical failure: recorded with its code if classified, or as
        # TECH.RUNTIME.UNCLASSIFIED if unexpected (memory, timeout...); the run is set to FAILED.
        if issues is not None:
            try:
                if isinstance(exc, ClassifiedFailure):
                    if exc.issue_code:
                        issues.emit(
                            exc.issue_code,
                            params=exc.params,
                            raw_error_class=type(exc.__cause__ or exc).__name__,
                            raw_message=str(exc.__cause__ or exc),
                            **exc.emit_fields,
                        )
                else:
                    issues.emit(
                        "TECH.RUNTIME.UNCLASSIFIED",
                        params={
                            "stage": active_step,
                            "raw_error_class": type(exc).__name__,
                        },
                        raw_error_class=type(exc).__name__,
                        raw_message=str(exc),
                    )
                issues.persist()
            except Exception:
                pass

        upsert_run_execution(run_row("FAILED", 0, 1, utc_now(), str(exc)[:4000]))
        raise


run_result = run_processing()
# Exit value for the notebook or batch that launches this one (run summary, as JSON).
mssparkutils.notebook.exit(json.dumps(run_result))
